# Xu 44-parameter forward model interface

This notebook is designed for OAT/GSA work. It keeps the Xu scientific kernels
from `xu_model_complete.ipynb`, then adds a grouped-parameter interface similar
to `moss_forward_model_143.ipynb`.

Document content is treated as reference material only; the executable request
is to expose a reproducible 44-dimensional forward model:

```text
parameter_values -> total_GFR_nl_min and diagnostics
```

In [ ]:
from pathlib import Path
import sys

from copy import deepcopy


def _resolve_workspace_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [
        start,
        start.parent,
        Path("/home/yg20/XU_GSA_OAT"),
    ]
    for candidate in candidates:
        if (
            (candidate / "range_search_xu_47" / "xu_44_parameter_ranges_literature.xlsx").exists()
            and (candidate / "Xu_model_complete" / "final_tree.vtk").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the Xu GSA workspace containing both range_search_xu_47/ "
        "and Xu_model_complete/final_tree.vtk"
    )


WORKSPACE_ROOT = _resolve_workspace_root()
PROJECT_ROOT = WORKSPACE_ROOT
MODEL_DIR = WORKSPACE_ROOT / "Xu_model_complete"

PARAMETER_TABLE_PATH = WORKSPACE_ROOT / "range_search_xu_47" / "xu_44_parameter_ranges_literature.xlsx"
PARAMETER_SHEET = "Parameter_Ranges_44"
PRIMARY_OUTPUT = "total_GFR_nl_min"

RUN_BASELINE_FULL_MODEL = False
DEFAULT_RUN_FULL_MODEL = False

OAT_OUTPUT_DIR = WORKSPACE_ROOT / "outputs" / "oat_44"
MODEL_OUTPUT_ROOT = WORKSPACE_ROOT / "outputs" / "xu_forward_model_44"
RUN_LABEL = None
for path in [OAT_OUTPUT_DIR, MODEL_OUTPUT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Fixed structure/model switches for the 44-D forward model.
USE_AUTOREGULATION = True
ONLY_MYOGENIC = False
USE_NEPHRON_POPULATIONS = True
SAVE_DETAILED_ITERATION_PLOTS = False

print("Workspace:", WORKSPACE_ROOT)
print("Model dir:", MODEL_DIR)
print("Parameter table:", PARAMETER_TABLE_PATH)
print("Primary output:", PRIMARY_OUTPUT)

## 1. Imports

In [ ]:
from __future__ import annotations

import concurrent.futures
import copy
import json
import logging
import os
import sys
import time
import warnings
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams
import networkx as nx
import nibabel as nib
import numpy as np
import pandas as pd
import pyvista
import scipy
import scipy.optimize as so
import scipy.spatial
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, fsolve
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
from tqdm.auto import tqdm

plt.rcParams.update({"figure.max_open_warning": 0, "figure.autolayout": True})


## 2. Inputs and all physiological parameters

Edit this section before running a case. `RUN_FULL_MODEL=False` prevents an accidental long full-scale run.

In [ ]:
# =========================== EDITABLE INPUTS ===========================
# The notebook is intentionally safe by default: definitions and audits run,
# but the ~29,600-nephron full solve starts only after RUN_FULL_MODEL=True.

P_IN_MMHG = 100
NUM_ITER = 25
LEARNING_RATE = 0.20
RELATIVE_FLOW_TOLERANCE = 2e-5

ONLY_MYOGENIC = False          # False = myogenic + TGF; True = block TGF
USE_NEPHRON_POPULATIONS = False
RUN_FULL_MODEL = False
SAVE_DETAILED_ITERATION_PLOTS = False

ROOT_LOCATION = [588, 217, 650]
VOXEL_SIZE_UM = 22.6
BLOOD_VISCOSITY = 3.6e-15      # N s um^-2
MAX_WORKERS = max(1, min(76, (os.cpu_count() or 1) // 10))

# Paths are resolved once in the setup cell: parameter tables live in
# WORKSPACE_ROOT, while VTK/segmentation model assets live in MODEL_DIR.
FINAL_TREE_FILE = MODEL_DIR / "final_tree.vtk"
SURFACE_FILE = MODEL_DIR / "surface.nii.gz"
PRECOMPUTED_VTK = MODEL_DIR / "data" / f"kirchhoff_p_in = {int(P_IN_MMHG)}.vtk"
OUTPUT_DIR = MODEL_OUTPUT_ROOT / f"{int(P_IN_MMHG)}mmHg_{'myogenic' if ONLY_MYOGENIC else 'full'}"

# Use the supplied pressure-specific VTK when available. It is an initial
# no-autoregulation state; every outer iteration still resolves the full tree.
TREE_FILE = PRECOMPUTED_VTK if PRECOMPUTED_VTK.exists() else FINAL_TREE_FILE

print({
    "tree_file": str(TREE_FILE),
    "surface_file": str(SURFACE_FILE),
    "P_in_mmHg": P_IN_MMHG,
    "mode": "myogenic only" if ONLY_MYOGENIC else "myogenic + TGF",
    "workers": MAX_WORKERS,
    "run_full_model": RUN_FULL_MODEL,
})


# ---------------- Published physiological parameters ----------------
@dataclass
class Glomerular:
    C_A: float = 57 #g/l
    H_A: float = 0.5
    Kf: float = 2.5
    R_E: float = 0.209
    R_PC: float = 0.0702
    a: float = 0.1631 # mmHg l g-1
    b: float = 0.00294
    L: float = 10

@dataclass
class Tubular:
    Km: float = 20 #mM
    Lv: float = 2e-5 # cm2 l osmol-1 s-1
    ns: float = 2
    Ls: float = 3.4e-7 #cm2 s-1
    V_max1 = 1e-7 # mmol cm-1 s-1
    V_max2 = 0.65e-7
    alpha: float = 1.65e-2 # (nl min-1 mmHg-5)1/4
    beta: float = 0.924  # (nl min-1 mmHg-1)1/4
    theta: float = 1.3 # cm-1
    keppa: float = 33.6 #nl min-1 cm-1
    gamma: float = 1.33e-5 # cm mmHg-1
    mu: float = 7.2e-4 #g/(cm s) # Ps s
    r_proximal: float = 12e-4 #cm
    r_loop: float = 10e-4 #cm
    Ls_long: float = 5.2e-7 #cm2 s-1
    Ls_inter: float = 5.6e-7 #cm2 s-1
    V_max1 = V_max1 * 1.22
    V_max2 = V_max2 * 1.22

@dataclass
class AA:

    sigma_e: float = 25.56 #dyn cm-2

    k_e: float = 17304 #cm-1
    r_e: float = 7 #micro-meter
    k_m: float = 8.75e6 # cm-2
    r_m: float = 12.5 #micro-meter

    sigma_m: float = 7.23e5 #dyn cm-2
    sigma_v: float = 1e6 # dyn s cm-2
    r_n: float = 10.04 # micro-meter no really usede
    h_init = 2 # micro-meter

    r_m = 14
    k_m = 3e6
    sigma_m = 10e5
    r_e = 5
    k_e = 1.2e4
    sigma_e = 6.4

@dataclass
class Myo:
    lam: float = 1
    # T0: float = 84.7 #dyn cm-1   #value from old paper
    T0: float = 81.1252  #dyn cm-1

    G: float = 0.06 # cm dyn -1 (in the range between 0.03 to 0.09)

@dataclass
class TGF:

    C_half: float = 46 # mmol/l
    k: float = (70 + 100)/2 * 1e-3 # l/mmol
    ita_max: float = 0.091
    phi: float = 0.182

@dataclass
class TGF_INTER:
    k: float = 0.8328 * (70 + 100)/2 * 1e-3 # l/mmol
    ita_max: float = 0.1183
    phi: float = 0.2184


@dataclass
class TGF_LONG:
    k: float = 0.05/0.07 * (70 + 100)/2 * 1e-3 # l/mmol
    ita_max: float = 0.1456
    phi: float = 0.2583


## 3. Glomerular and tubular equations

In [ ]:
def C_protein(Q_A, P_GC, P_T0, x, C):
    return Glomerular.Kf/(Glomerular.L * Q_A * Glomerular.C_A) * C**2 * (
            P_GC - P_T0 - Glomerular.a * C - Glomerular.b * C**2)

def P_T_desc(sol_desc, z, P):
    Qz = sol_desc(z)[0]
    dP = - 1/60 * 1/133.322 * 1e-6 * 8 * Tubular.mu/(np.pi * Tubular.r_loop**4) * Qz
    return dP

def Q_T_proximal(Q_T0, z):
    C = Q_T0 - Tubular.keppa/Tubular.theta
    Q_T = C + Tubular.keppa/Tubular.theta * np.exp(-Tubular.theta * z)
    return Q_T

def Combined_desc(z, F):
    C_I = 150 + 150/0.3 * z
    Q_T, C = F
    dQ_T = - Tubular.Lv * Tubular.ns * (C_I - C) * 6e4 #  cm2 l osmol-1 s-1 * mmol =
    dC = (-C * dQ_T - 6e7 * Tubular.Ls * (C - C_I))/Q_T
    return dQ_T, dC


def Combined_desc_long_inter(z, F):
    C_I = 150 + 450/0.8 * z
    Q_T, C = F
    dQ_T = - Tubular.Lv * Tubular.ns * (C_I - C) * 6e4 #  cm2 l osmol-1 s-1 * mmol =
    dC = (-C * dQ_T - 6e7 * Tubular.Ls * (C - C_I))/Q_T
    return dQ_T, dC


def Cs_asce(Q_T_desc_end, z, C, md_loc=0.5):
    C_I = (300 - 150/0.3 * z) if z < 0.3 else 150
    Vmax = Tubular.V_max1 if z < md_loc else Tubular.V_max2
    return - 6e7 * (Tubular.Ls * (C - C_I) + 1e3 * Vmax * C/(Tubular.Km + C)
             )/Q_T_desc_end

def Cs_asce_thin(Q_T_desc_end, z, C, long=True):
    C_I = 300
    Ls  = Tubular.Ls_long if long else Tubular.Ls_inter
    return - 6e7 * (Ls * (C - C_I))/Q_T_desc_end


def P_T_asce(Q_T_desc_end, P_end, z, asce_length=0.65):
    return (1/60 * 1/133.322 * 1e-6 * 8 * Tubular.mu/(np.pi * Tubular.r_loop**4) *
            Q_T_desc_end * (asce_length - z) + P_end)

def P_T_proximal(Q_T0, P_TZ, z):
    Z = 1
    k2 = 1/60 * 1/133.322 * 1e-6 * 8 * Tubular.mu/(np.pi * Tubular.r_proximal**4)
    k1 = k2 * Tubular.keppa/Tubular.theta**2
    P_T = (k1 * (np.exp(-Tubular.theta * z) - np.exp(-Tubular.theta * Z)) +
           k2 * (Q_T0 - Tubular.keppa/Tubular.theta) * (Z - z) + P_TZ)
    return P_T


## 4. Nephron solver

This cell contains the nonlinear glomerular solve, tubular ODE integrations, and tubular-pressure consistency solve.

In [ ]:
class NegativeFlowException(Exception):
    pass


def func_Q_T0(Q_T0, Q_A, P_T0, P_GC=None):

    if P_GC is None:
        P_GC = Glomerular.R_E * (Q_A / (1 - Glomerular.H_A) - Q_T0) + Glomerular.R_PC * Q_A / (1 - Glomerular.H_A)

    C_x = partial(C_protein, Q_A, P_GC, P_T0)

    sol = solve_ivp(C_x, [0, Glomerular.L], [Glomerular.C_A],  method='Radau', dense_output=True)
    if not sol.success:
        print(f'error in solving C_x, {sol.message = } with {Q_A = }, {P_GC = }, {P_T0 = }')
        logging.warning(f'error in solving C_x, {sol.message = } with {Q_A = }, {P_GC = }, {P_T0 = }')
        raise RuntimeWarning

    C_E = np.squeeze(sol.y[:, -1])

    Q_T0_new = (1 - Glomerular.C_A / C_E) * Q_A

    return Q_T0 - Q_T0_new



def func_glomerular(P_T0, Q_A, P_GC=None, final=False, long=False, inter=False, debug=False):

    assert not (long and inter)
    Cs_0 = 150
    Q_T0 = Q_A/3

    if not np.isscalar(Q_A):
        Q_T0 = np.ones(len(Q_A)) * Q_T0
        Cs_0 = np.ones(len(Q_A)) * Cs_0

    Cs_list = []
    t_list = []

    Q_T0 = fsolve(func_Q_T0, x0=Q_T0, args=(Q_A, P_T0, P_GC))

    if P_GC is None:
        P_GC = Glomerular.R_E * (Q_A / (1 - Glomerular.H_A) - Q_T0) + Glomerular.R_PC * Q_A / (1 - Glomerular.H_A)

    if np.isscalar(Q_A):
        assert len(Q_T0) == 1
        Q_T0 = Q_T0[0]
    else:
        assert len(Q_T0) == len(Q_A)

    ############    Proximal part    ############

    Q_T_proximal_end = Q_T_proximal(Q_T0, z=1)

    if Q_T_proximal_end < 0:
        raise NegativeFlowException('negative in Q_T_proximal_end')

    ############   descending limb    ############

    if long:
        sol = solve_ivp(Combined_desc_long_inter, [0, 0.8], [Q_T_proximal_end, Cs_0],
                        method='Radau', dense_output=True)
    elif inter:
        sol = solve_ivp(Combined_desc_long_inter, [0, 0.55], [Q_T_proximal_end, Cs_0],
                        method='Radau', dense_output=True)
    else:
        sol = solve_ivp(Combined_desc, [0, 0.3], [Q_T_proximal_end, Cs_0],
        method='Radau', dense_output=True)

    if not sol.success:
        print(f'error in solving desc, {sol.message = } with {Q_A = }, {P_GC = }, {Q_T_proximal_end = }')
        logging.warning(f'error in solving desc, {sol.message = } with {Q_A = }, {P_GC = }, {Q_T_proximal_end = }')
        raise RuntimeWarning

    z_desc_span = np.squeeze(sol.t)
    Q_T_desc_list = sol.y[0]
    if debug:
        Cs_desc_list = sol.y[1]
        Cs_list.append(Cs_desc_list)
        t_list.append(z_desc_span)

    Q_T_desc_end = np.squeeze(sol.y)[0, -1]
    Cs_desc_end = np.squeeze(sol.y)[1, -1]

    sol_desc = sol.sol # this is required to solve P_T backwards

    if Q_T_desc_end < 0:
        raise NegativeFlowException('negative in Q_T_desc_end')
    if Cs_desc_end < 0:
        raise NegativeFlowException('negative in Cs_desc_end')

    ############   ascending limb    ############

    if long or inter:

        Cs_asce_thin_cur = partial(Cs_asce_thin, Q_T_desc_end, long=long)

        asce_thin_len = 0.5 if long else 0.25

        sol = solve_ivp(Cs_asce_thin_cur, [0, asce_thin_len],
                        [Cs_desc_end],
                        method='Radau', dense_output=True
                        )

        if not sol.success:
            print(f'error in solving thin asce, {sol.message = } with {Q_A = }, {P_GC = }, {Q_T_proximal_end = }, '
                  f'{Cs_desc_end = }, {Cs_desc_end = }')
            logging.warning(f'error in solving thin desc, {sol.message = } with {Q_A = }, {P_GC = }, {Q_T_proximal_end = }, '
                            f'{Cs_desc_end = }, {Cs_desc_end = }')
            raise RuntimeWarning

        Cs_thin_end = np.squeeze(sol.y)[-1]

        if debug:
            Cs_list.append(np.squeeze(sol.y))
            t_list.append(np.squeeze(sol.t) + t_list[-1][-1])
        if long:
            z_range = np.concatenate([np.arange(0, 0.3, 0.001),
                                      np.arange(0.3, 0.32 - 1e-6, 0.001),
                                      np.arange(0.32, 0.47 - 1e-6, 0.001)]
                                     )
        else:
            z_range = np.concatenate([np.arange(0, 0.3, 0.001),
                                      np.arange(0.3, 0.4 - 1e-6, 0.001),
                                      np.arange(0.4, 0.55 - 1e-6, 0.001)]
                                     )

    else:
        z_range = np.concatenate([np.arange(0, 0.3, 0.001),
                                  np.arange(0.3, 0.5 - 1e-6, 0.001),
                                  np.arange(0.5, 0.65 - 1e-6, 0.001)]
                                 )

    if long or inter:
        asce_len = 0.47 if long else 0.55
        md_loc = 0.32 if long else 0.4

        Cs_asce_cur = partial(Cs_asce, Q_T_desc_end, md_loc=md_loc)

        sol = solve_ivp(Cs_asce_cur, [0, asce_len],
                        [Cs_thin_end], t_eval=z_range,
                        method='Radau', dense_output=True
                        # method='LSODA',
                        )
    else:
        Cs_asce_cur = partial(Cs_asce, Q_T_desc_end)
        sol = solve_ivp(Cs_asce_cur, [0, 0.65],
                        [Cs_desc_end], t_eval=z_range,
                        method='Radau', dense_output=True
                        # method='LSODA',
                        )


    if not sol.success:
        print(f'error in solving asce, {sol.message = } with {Q_A = }, {P_GC = }, {Q_T_proximal_end = }, '
              f'{Cs_desc_end = }, {Cs_desc_end = }')
        logging.warning(f'error in solving desc, {sol.message = } with {Q_A = }, {P_GC = }, {Q_T_proximal_end = }, '
                        f'{Cs_desc_end = }, {Cs_desc_end = }')
        raise RuntimeWarning

    if long:
        md_index = np.argmin(np.abs(z_range - 0.32))
    elif inter:
        md_index = np.argmin(np.abs(z_range - 0.4))
    else:
        md_index = np.argmin(np.abs(z_range - 0.5))

    y = np.squeeze(sol.y)
    t = np.squeeze(sol.t)
    assert len(y) == len(z_range)
    Cs_md = y[md_index]

    Cs_end = y[-1]

    if Cs_md < 0:
        raise NegativeFlowException('negative in Cs_md')

    if debug:
        off_set = t_list[-1][-1]
        Cs_list.append(y[:md_index])
        t_list.append(t[:md_index] + off_set)
        Cs_list.append(y[md_index+1:])
        t_list.append(t[md_index+1:] + off_set)


    logging.debug(f'{Cs_0 = }')
    logging.debug(f'{Cs_desc_end = }')
    logging.debug(f'{Cs_md = }')
    logging.debug(f'{Cs_end = }')

    ############   solve P_T at the end
    '''
    note Q is constant at ascending, so Q_T_asce_end = Q_T_desc_end
    '''

    sol = fsolve(lambda x: x - Q_T_desc_end / (Tubular.alpha * x + Tubular.beta) ** 4, x0=7.2467)

    assert len(sol) == 1

    P_end = sol[0]

    logging.debug(f'{P_end = }')

    if long:
        asce_length = 0.97
    elif inter:
        asce_length = 0.8
    else:
        asce_length = 0.65

    ############   solve P_T at asce backwards    ############

    P_desc_end = P_T_asce(Q_T_desc_end, P_end, 0, asce_length=asce_length)

    logging.debug(f'{P_desc_end = }')

    P_md = P_T_asce(Q_T_desc_end, P_end, asce_length-0.15, asce_length=asce_length)
    logging.debug(f'{P_md = }')

    ############   solve P_T at P desc  backwards    ############

    dP_desc_list = Q_T_desc_list[1:] * np.diff(z_desc_span) * 1 / 60 * 1 / 133.322 * 1e-6 * 8 * Tubular.mu / (
                np.pi * Tubular.r_loop ** 4)

    P_desc_list = P_desc_end + np.cumsum(dP_desc_list[::-1])
    P_proximal_end_est = P_desc_list[-1]
    logging.debug(f'{P_proximal_end_est = }')


    P_T_desc_cur = partial(P_T_desc, sol_desc)

    if long:
        desc_len = 0.8
    elif inter:
        desc_len = 0.55
    else:
        desc_len = 0.3

    sol = solve_ivp(P_T_desc_cur, [desc_len, 0],
                    [P_desc_end], method='LSODA',)

    P_proximal_end = np.squeeze(sol.y)[-1]

    logging.debug(f'{P_proximal_end = }')

    P_start = P_T_proximal(Q_T0, P_proximal_end, z=0)

    logging.debug(f'{P_start = }')

    if not final:
        return P_T0 - P_start

    else:
        if debug:
            return P_T0, Cs_md, Q_T0, Cs_desc_end, Q_T_desc_end, Cs_list, t_list
        else:
            return P_T0, Cs_md, Q_T0, Cs_desc_end, Q_T_desc_end, P_end, P_md


## 5. Afferent arteriole: myogenic response, TGF and equilibrium radius

In [ ]:
def fx(x_tgf, x_myo):

    # x = x_tgf + x_myo

    x = x_myo

    fx = 3 * np.exp(x) / (np.exp(x) + 2 * np.exp(-0.5 * x))

    return fx

def h_v(r_v):
    r_0 = 10
    h_0 = 2
    return -r_v + np.sqrt(r_v**2 + 2 * h_0 * r_0 + h_0**2)

def AA_model(r_num, Cs_md, P_v, final=False, only_myo=False, type=0):

    if type == 0:
        x_tgf = 5 * 1 / Myo.lam * (TGF.ita_max -
                               TGF.phi / (1 + np.exp(TGF.k * (Cs_md - TGF.C_half)))
                               )
    elif type == 1:
        x_tgf = 5 * 1 / Myo.lam * (TGF_INTER.ita_max -
                               TGF_INTER.phi / (1 + np.exp(TGF_INTER.k * (Cs_md - TGF.C_half)))
                               )
    elif type == 2:
        x_tgf = 5 * 1 / Myo.lam * (TGF_LONG.ita_max -
                               TGF_LONG.phi / (1 + np.exp(TGF_LONG.k * (Cs_md - TGF.C_half)))
                               )
    else:
        raise Exception("Nephron type must be either 0, 1, or 2")

    x_tgf = 0 if only_myo else x_tgf

    x_myo = Myo.lam * Myo.G * (r_num * P_v / (1e3 / 133.32) - Myo.T0 *
                                   (1 - x_tgf))

    T_e = 1e-4 * AA.sigma_e * h_v(r_num) * (
               np.exp(1e-4 * AA.k_e * (r_num - AA.r_e)) - 1)  # dyn/cm2 * um = 10-4 dyn/cm
    T_m = 1e-4 * AA.sigma_m * h_v(r_num) * fx(x_tgf, x_myo) * np.exp(-1e-8 * AA.k_m * (r_num - AA.r_m) ** 2)

    if not final:
        return r_num * P_v / (1e3 / 133.32) - T_e - T_m
    else:
        return r_num * P_v / (1e3 / 133.32), x_myo, x_tgf, T_e, T_m


## 6. Reconstructed vascular tree and released Kirchhoff implementation

The complete released vascular-tree classes are embedded here for transparency. The next cell installs the equation-identical sparse implementation used for full-scale execution.

In [ ]:
# matplotlib.use('Agg')


class VtkNetwork:

    def __init__(self, pt_file=None, root_loc=None, vsize=22.6):

        self.pt_file = pt_file
        self.root_loc = root_loc
        self.vsize = vsize

    def build(self):
        mesh = pyvista.read(self.pt_file,
                                      )
        points = mesh.points

        line = mesh.lines

        if 'radius' in mesh.point_data:
            node_radius = mesh.point_data['radius']
        elif 'node_radius' in mesh.point_data:
            node_radius = mesh.point_data['node_radius']
        elif 'radius' in mesh.cell_data:
            node_radius = None

        self.tree = nx.Graph()

        for i, p in enumerate(points):
            self.tree.add_node(i, loc=np.array(p), fixed=False, root=False,
                               node_radius=node_radius[i] if node_radius is not None else None
                               )

        i = 1
        while i < len(line):
            node1, node2 = line[i], line[i + 1]
            cur_dict = {k: v[i//3] for k, v in mesh.cell_data.items()}
            self.tree.add_edge(node1, node2, **cur_dict)
            i += 3

        for edge in self.tree.edges:
            a, b = edge
            self.tree[a][b]['radius_neg'] = - self.tree[a][b]['radius']

        if self.root_loc is not None:
            self.root = self.find_n_with_coord(self.root_loc)
            self.tree.nodes[self.root]['fixed'] = True
            self.tree.nodes[self.root]['root'] = True

        Gcc = sorted(nx.connected_components(self.tree), key=len, reverse=True)
        self.tree = nx.Graph(self.tree.subgraph(Gcc[0]))

        self.update_order()

    def mst(self):

        tree = nx.minimum_spanning_tree(self.tree, weight='radius_neg')
        root = self.root

        self.tree = tree
        self.root = self.find_n_with_coord(self.root_loc)

        self.tree.nodes[root]['fixed'] = True
        self.tree.nodes[root]['root'] = True

        self.update_order()
        self.to_directed()


        # di_tree = tree.to_directed()

    def to_directed(self):
        def bfs_to_direct(node):
            neighbors = list(self.tree.neighbors(node))

            if self.tree.nodes[node]['root']:
                children = neighbors
            else:
                neighbor_order = np.array([self.tree.nodes[n]['level'] for n in neighbors])
                if -1 in neighbor_order:
                    root_idx = np.where(neighbor_order == -1)[0][0]
                else:
                    root_idx = np.argmax(neighbor_order)

                root_idx = neighbors[root_idx]

                children = [i for i in neighbors if i != root_idx]

            if len(children) == 0:
                return

            for child in children:
                attrs = self.tree[node][child]
                self.di_tree.add_edge(node, child, **attrs)

            for child in children:
                bfs_to_direct(child)

        self.di_tree = nx.DiGraph()

        for n in self.tree.nodes:
            attrs = self.tree.nodes[n]
            self.di_tree.add_node(n, **attrs)

        root = [i for i in self.tree.nodes if self.tree.nodes[i]['root']][0]

        bfs_to_direct(root)


    def get_depth(self, root=None):
        if root is None:
            all_nodes = np.array([i for i in self.tree.nodes])

            orders = np.array([self.tree.nodes[n]['level'] for n in all_nodes])

            root = all_nodes[np.where(orders == -1)[0][0]]

            neighbors = np.array(list(self.tree.neighbors(root)))

            depth = 1 + max([self.get_depth(i) for i in neighbors])
            self.tree.nodes[root]['depth'] = depth
            return depth

        neighbors = np.array(list(self.tree.neighbors(root)))
        if len(neighbors) == 1:
            self.tree.nodes[root]['depth'] = 1
            return 1
        else:
            neighbor_orders = np.array([self.tree.nodes[n]['level'] for n in neighbors])
            if -1 in neighbor_orders:
                root_idx = np.where(neighbor_orders == -1)[0][0]
            else:
                root_idx = np.argmax(neighbor_orders)

            depth = 1 + max([self.get_depth(self.tree, i) for i in neighbors if i != neighbors[root_idx]])
            self.tree.nodes[root]['depth'] = depth
            return depth

    def save(self, file='sanity_check3.vtk'):

        for i in self.tree.nodes:
            self.tree.nodes[i]['ind'] = i

        nodes_inds = self.tree.nodes
        indices = sorted(nodes_inds)

        points = [nodes_inds[i]['loc'] for i in indices]
        points = np.array(points)

        edges_inds = self.tree.edges
        edges_inds = np.array(edges_inds)

        radius_list = [self.tree[n1][n2]['radius'] for n1, n2 in edges_inds]
        radius_list = np.array(radius_list)

        if np.max(edges_inds) >= len(nodes_inds):
            map = dict(zip(indices, np.arange(len(indices))))
            edges_inds_new = np.vectorize(map.get)(edges_inds)
        else:
            edges_inds_new = edges_inds

        lines = np.hstack([np.ones((len(edges_inds_new), 1), dtype=np.uint8) * 2, edges_inds_new])
        lines = np.hstack(lines)
        mesh = pyvista.PolyData(points,
                                lines=lines
                                )

        for key in self.tree[edges_inds[0][0]][edges_inds[0][1]]:
            edge_feature = [self.tree[n1][n2][key] for n1, n2 in edges_inds]
            edge_feature = np.array(edge_feature)

            assert len(edge_feature) > 0 and len(edge_feature) == mesh.n_lines
            mesh.cell_data[key] = edge_feature

        for key in self.tree.nodes[indices[0]]:
            if key in ('loc', 'fixed', 'root'):
                continue
            node_feature = [self.tree.nodes[i][key] for i in indices]
            assert len(node_feature) > 0 and len(node_feature) == mesh.n_points
            mesh.point_data[key] = np.array(node_feature)

        mesh.save(file)

    def update_order(self, mode='level'):
        for n in self.tree.nodes:
            self.tree.nodes[n][mode] = 1 if self.tree.degree[n] <= 1 and not self.tree.nodes[n]['fixed'] else 0
            if self.tree.nodes[n]['fixed']:
                self.tree.nodes[n][mode] = -1
        count_no_label = len(self.tree.nodes)
        while count_no_label != 0:
            for node in self.tree.nodes:
                if self.tree.nodes[node][mode] != 0 or len(list(self.tree.neighbors(node))) == 0:
                    continue
                neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])
                if -1 in neighbor_orders and np.count_nonzero(neighbor_orders == 0) >= 1:
                    continue
                if -1 not in neighbor_orders and np.count_nonzero(neighbor_orders == 0) > 1:
                    continue
                max_order = np.max(neighbor_orders)
                max_count = np.count_nonzero(neighbor_orders == max_order)


                self.tree.nodes[node][mode] = max_order if max_count == 1 and mode == 'HS' else max_order + 1

            count_no_label = np.count_nonzero(np.array([self.tree.nodes[n][mode] for n in self.tree.nodes]) == 0)

    def update_final_order(self, mode='HS'):
        """
        Update order when no further operations will be applied.
        """

        for n in self.tree.nodes:
            self.tree.nodes[n][mode] = 1 if self.tree.degree[n] == 1 else 0
        count_no_label = len(self.tree.nodes)
        while count_no_label != 0:
            for node in self.tree.nodes:
                if self.tree.nodes[node][mode] != 0 or len(list(self.tree.neighbors(node))) == 0:
                    continue
                neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])

                if np.count_nonzero(neighbor_orders == 0) > 1:  # has to be second endpoint
                    continue
                max_order = np.max(neighbor_orders)
                max_count = np.count_nonzero(neighbor_orders == max_order)
                self.tree.nodes[node][mode] = max_order if max_count == 1 and mode == 'HS' else max_order + 1
            count_no_label = np.count_nonzero(np.array([self.tree.nodes[n][mode] for n in self.tree.nodes]) == 0)

        node = [n for n in self.tree.nodes if self.tree.nodes[n][mode] == -1][0]

        neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])
        max_order = np.max(neighbor_orders)
        max_count = np.count_nonzero(neighbor_orders == max_order)
        self.tree.nodes[node][mode] = max_order if max_count == 1 and mode == 'HS' else max_order + 1

    def sum_children_distances(self, n):

        neighbors = np.array(list(self.tree.neighbors(n)))
        neighbor_orders = np.array([self.tree.nodes[n]['level'] for n in neighbors])
        root = -1 if -1 in neighbor_orders else np.max(neighbor_orders)
        children = np.array([neighbors[i] for i in range(len(neighbors)) if neighbor_orders[i] != root])

        root = neighbors[np.where(neighbor_orders == root)[0][0]]
        if len(neighbors) == 1:
            return np.linalg.norm(self.tree.nodes[root]['loc'] - self.tree.nodes[n]['loc'])

        cur = 0

        for i in children:
            cur += self.sum_children_distances(i)

        return cur

    def label_dist_from_root(self, node=None):
        if node is None:
            node = [n for n in self.tree.nodes if self.tree.nodes[n]['root']][0]
            self.tree.nodes[node]['sub_length_root'] = 0
            children = np.array(list(self.tree.neighbors(node)))
        else:
            neighbors = np.array(list(self.tree.neighbors(node)))
            neighbor_orders = np.array([self.tree.nodes[n]['level'] for n in neighbors])
            root = -1 if -1 in neighbor_orders else np.max(neighbor_orders)
            children = np.array([neighbors[i] for i in range(len(neighbors)) if neighbor_orders[i] != root])
            root = neighbors[np.where(neighbor_orders == root)[0][0]]
            self.tree.nodes[node]['sub_length_root'] = self.tree.nodes[root]['sub_length_root'] + np.linalg.norm(
                self.tree.nodes[root]['loc'] - self.tree.nodes[node]['loc'])

        for i in children:
            self.label_dist_from_root(i)

    def label_dist_to_end(self, node=None):
        if node is None:
            node = [n for n in self.tree.nodes if self.tree.nodes[n]['root']][0]
            self.tree.nodes[node]['sub_length_end'] = 0
            children = np.array(list(self.tree.neighbors(node)))
        else:
            neighbors = np.array(list(self.tree.neighbors(node)))
            neighbor_orders = np.array([self.tree.nodes[n]['level'] for n in neighbors])
            root = -1 if -1 in neighbor_orders else np.max(neighbor_orders)
            children = np.array([neighbors[i] for i in range(len(neighbors)) if neighbor_orders[i] != root])
            root = neighbors[np.where(neighbor_orders == root)[0][0]]
            self.tree.nodes[node]['sub_length_root'] = self.tree.nodes[root]['sub_length_root'] + np.linalg.norm(
                self.tree.nodes[root]['loc'] - self.tree.nodes[node]['loc'])

        for i in children:
            self.label_dist_from_root(i)


    def max_children_distances(self, n):

        neighbors = np.array(list(self.tree.neighbors(n)))
        neighbor_orders = np.array([self.tree.nodes[n]['level'] for n in neighbors])
        root = -1 if -1 in neighbor_orders else np.max(neighbor_orders)
        children = np.array([neighbors[i] for i in range(len(neighbors)) if neighbor_orders[i] != root])

        root = neighbors[np.where(neighbor_orders == root)[0][0]]
        if len(neighbors) == 1:
            return np.linalg.norm(self.tree.nodes[root]['loc'] - self.tree.nodes[n]['loc'])

        cur = np.max([self.max_children_distances(i) for i in children]) + np.linalg.norm(
            self.tree.nodes[root]['loc'] - self.tree.nodes[n]['loc'])

        return cur

    def save_results(self, work_dir):
        """
        Stores the resulting network structure.
        """

        if not os.path.exists(work_dir):
            os.mkdir(work_dir)

        coord_file = os.path.join(work_dir, 'test_1_result_coords.npy')
        connection_file = os.path.join(work_dir, 'test_1_result_connections.npy')
        radius_file = os.path.join(work_dir, 'test_1_result_radii.npy')
        order_file = os.path.join(work_dir, 'test_1_result_HS_order.npy')
        level_file = os.path.join(work_dir, 'test_1_result_level_order.npy')

        nodes = dict()
        coords = list()
        connections = list()
        radii = list()
        order = list()
        l_order = list()
        # self.update_final_order('HS')
        # self.update_final_order('level')

        for edge in list(self.tree.edges):
            node1, node2 = edge
            for node in edge:
                if not node in nodes:
                    nodes[node] = len(coords)
                    coords.append(self.tree.nodes[node]['loc'])
                    order.append(self.tree.nodes[node]['HS'])
                    l_order.append(self.tree.nodes[node]['level'])
            connections.append([nodes[node1], nodes[node2]])
            radii.append(abs(self.tree[node1][node2]['radius']))

        np.save(coord_file, coords)
        np.save(connection_file, connections)
        np.save(radius_file, radii)
        print("Save coords, edges and radius.")
        np.save(order_file, order)
        np.save(level_file, l_order)
        print("Save orders.")

    def find_n_with_coord(self, coords=np.array([1, 2, 3])):
        all_nodes = [i for i in self.tree.nodes]
        all_loc = np.array([self.tree.nodes[i]['loc'] for i in all_nodes])
        all_dist = np.sum((all_loc - np.array(coords)) ** 2, axis=1)
        return all_nodes[np.argmin(all_dist)]

    def remove_intermediate(self):

        max_orders = np.max(np.array([self.tree.nodes[n]['level'] for n in self.tree.nodes]))
        root = [n for n in self.tree.nodes if self.tree.nodes[n]['level'] == max_orders]
        assert len(root) == 1
        root = root[0]
        all_nodes = [n for n in self.tree.nodes]
        for n in all_nodes:
            neighbors = list(self.tree.neighbors(n))
            if len(neighbors) == 2 and n != root:
                left = neighbors[0]
                right = neighbors[1]

                dis = np.linalg.norm(self.tree.nodes[left]['loc'] - self.tree.nodes[right]['loc'])
                r = np.mean([self.tree[left][n]['radius'], self.tree[n][right]['radius']])
                self.tree.add_edge(left, right, radius=r, radius_neg=-r, length=dis)

                self.tree.remove_node(n)

        self.update_order()

        return


    def make_node_radius(self):

        for n in self.tree.nodes:
            self.tree.nodes[n]['node_radius'] = 0

        for n in self.tree.edges:
            a, b = n[0], n[1]
            self.tree.nodes[b]['node_radius'] = max(self.tree[a][b]['radius'], self.tree.nodes[b]['node_radius'])
            self.tree.nodes[a]['node_radius'] = max(self.tree[a][b]['radius'], self.tree.nodes[a]['node_radius'])

        root = [i for i in self.tree.nodes if self.tree.nodes[i]['node_radius'] == 0]
        print(len(root))

        for n in self.tree.edges:
            for i in n:
                self.tree.nodes[i]['node_radius_scaled'] = self.tree.nodes[i]['node_radius'] / self.vsize


    def rescale_new_terminals(self, mu=10.04, sigma=0.14, flow=None):
        for n in self.tree.nodes:
            if self.tree.nodes[n]['level'] != 0 and len(list(self.tree.neighbors(n))) == 0:
                root = list(self.tree.predecessors(n))[0]
                self.tree[root][n]['radius'] = np.random.normal(loc=mu, scale=sigma)
                self.tree[root][n]['flow'] = flow

    def rescale_fixed_radius_flow(self, mode='level'):

        for n in self.tree.nodes:
            self.tree.nodes[n][mode] = 1 if self.tree.degree[n] <= 1 else 0
            if self.tree.nodes[n]['root']:
                self.tree.nodes[n][mode] = -1

        count_no_label = len(self.tree.nodes)


        while count_no_label != 0:
            for node in self.tree.nodes:
                if self.tree.nodes[node][mode] != 0 or len(list(self.tree.neighbors(node))) == 0:
                    continue

                neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])

                root = list(self.tree.predecessors(node))[0]

                if np.count_nonzero(neighbor_orders == 0) >= 1:
                    continue

                max_order = np.max(neighbor_orders)
                self.tree.nodes[node][mode] = max_order + 1

                r_child = np.array([self.tree[node][n]['radius'] for n in self.tree.neighbors(node)])

                f_child = np.array([self.tree[node][n]['flow'] for n in self.tree.neighbors(node)])

                r_root = np.power(np.sum(r_child ** 3), 1 / 3)
                f_root = np.sum(f_child)

                self.tree[root][node]['radius'] = r_root
                self.tree[root][node]['flow'] = f_root

            count_no_label = np.count_nonzero(np.array([self.tree.nodes[n][mode] for n in self.tree.nodes]) == 0)


class VtkNetworkAnalysis(VtkNetwork):

    def __init__(self, pt_file=None, root_loc=None, vsize=22.6, mu=0, t_flow=0):
        super().__init__(pt_file, root_loc, vsize)

        self.mu = mu

    def build(self):

        mesh = pyvista.read(self.pt_file)

        points = mesh.points
        line = mesh.lines

        self.tree = nx.Graph()

        point_data = mesh.point_data
        cell_data = mesh.cell_data

        point_feature_all = {key: value for (key, value) in point_data.items()}

        for i, p in enumerate(points):

            point_feature = {key: value[i] for (key, value) in point_feature_all.items()}

            self.tree.add_node(i, loc=np.array(p),
                               root=False, fixed=False,
                               **point_feature
                               )

        i = 1
        while i < len(line):
            node1, node2 = line[i], line[i + 1]

            edge_feature = {}
            for key in cell_data.keys():
                edge_feature[key] = cell_data[key][i // 3]

            self.tree.add_edge(node1, node2,
                               **edge_feature,
                               )
            i += 3

        for edge in self.tree.edges:
            a, b = edge
            self.tree[a][b]['radius_neg'] = - self.tree[a][b]['radius']

        all_nodes = [i for i in self.tree.nodes]

        if 'level' in self.tree[all_nodes[0]]:
            all_levels = np.array([self.tree.nodes[i]['level'] for i in all_nodes])
            if -1 in all_levels:
                root_idx = np.where(all_levels == -1)[0][0]
            else:
                root_idx = np.argmax(all_levels)

            self.root = all_nodes[root_idx]

        elif self.root_loc is not None:
            self.root = self.find_n_with_coord(self.root_loc)

        self.tree.nodes[self.root]['root'] = True
        self.tree.nodes[self.root]['fixed'] = True

        Gcc = sorted(nx.connected_components(self.tree), key=len, reverse=True)
        self.tree = nx.Graph(self.tree.subgraph(Gcc[0]))

        self.remove_intermediate()

        self.update_order()

        self.node_count = len(list(self.tree.nodes))

    def add_vessel(self, node, neighbor_node, radius=None, flow=None, sub=-1):
        """
        Adds a vessel between two nodes.

        Parameters
        --------------------
        node -- one endpoint
        neighbor_node -- the other endpoint
        """

        r = radius
        f = flow
        dis = np.linalg.norm(self.tree.nodes[node]['loc'] - self.tree.nodes[neighbor_node]['loc']
                             ) * self.vsize

        self.tree.add_edge(node, neighbor_node, radius=r, flow=f, length=dis, sub=sub)

    def remove_intermediate(self):
        max_orders = np.max(np.array([self.tree.nodes[n]['level'] for n in self.tree.nodes]))
        root = [n for n in self.tree.nodes if self.tree.nodes[n]['level'] == max_orders]
        assert len(root) == 1
        root = root[0]
        all_nodes = [n for n in self.tree.nodes]
        for n in all_nodes:
            neighbors = list(self.tree.neighbors(n))
            if len(neighbors) == 2 and n != root:
                left = neighbors[0]
                right = neighbors[1]

                edge_feature = {}
                for key in self.tree[left][n].keys():
                    edge_feature[key] = self.tree[left][n][key]

                self.tree.add_edge(left, right, **edge_feature)
                self.tree.remove_node(n)

    def remove_intermediate_di(self):

        all_nodes = [n for n in self.tree.nodes]
        nodes_to_remove = []
        for n in all_nodes:
            neighbors = list(self.tree.successors(n))

            if len(neighbors) == 1 and n != self.root:
                left = list(self.tree.predecessors(n))[0]
                right = neighbors[0]
                edge_feature = {}
                for key in self.tree[left][n].keys():
                    edge_feature[key] = self.tree[left][n][key]

                self.tree.add_edge(left, right, **edge_feature)

                nodes_to_remove.append(n)

            self.tree.remove_nodes_from(nodes_to_remove)

    def update_order_di(self, mode='level'):
        for n in self.tree.nodes:

            self.tree.nodes[n][mode] = 1 if self.tree.degree[n] <= 1 else 0

            if self.tree.nodes[n]['fixed'] and self.tree.nodes[n]['root']:
                self.tree.nodes[n][mode] = -1

        count_no_label = self.node_count
        while count_no_label != 0:
            for node in self.tree.nodes:
                if self.tree.nodes[node][mode] != 0 or len(list(self.tree.neighbors(node))) == 0:
                    continue
                neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])

                root = list(self.tree.predecessors(node))[0]

                if np.count_nonzero(neighbor_orders == 0) >= 1:
                    continue

                max_order = np.max(neighbor_orders)
                max_count = np.count_nonzero(neighbor_orders == max_order)

                if len(neighbor_orders) == 1:
                    self.tree.nodes[node][mode] = max_order
                else:
                    self.tree.nodes[node][mode] = max_order if max_count == 1 and mode == 'HS' else max_order + 1

            count_no_label = np.count_nonzero(np.array([self.tree.nodes[n][mode] for n in self.tree.nodes]) == 0)

    def depth_order(self):
        for n in self.tree.nodes:
            self.tree.nodes[n]['depth'] = -1
        for n in self.tree.nodes:
            if self.tree.nodes[n]['depth'] != -1:
                continue
            else:
                self.tree.nodes[n]['depth'] = self.find_depth(n)

    def find_depth(self, n):
        root = list(self.tree.predecessors(n))
        if len(root) == 0:
            return 0
        else:
            root = root[0]
            return 1 + self.find_depth(root)



    def update_final_order_di(self, mode='HS'):
        """
        Update order when no further operations will be applied.
        """

        for n in self.tree.nodes:
            self.tree.nodes[n][mode] = 1 if self.tree.degree[n] == 1 else 0
        count_no_label = self.node_count
        cur_order = 1
        while count_no_label != 0:
            for node in self.tree.nodes:
                if self.tree.nodes[node][mode] != 0 or len(list(self.tree.neighbors(node))) == 0:
                    continue
                neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])

                if np.count_nonzero(neighbor_orders == 0) >= 1:  # has to be second endpoint
                    continue
                max_order = np.max(neighbor_orders)
                max_count = np.count_nonzero(neighbor_orders == max_order)
                self.tree.nodes[node][mode] = max_order if max_count == 1 and mode == 'HS' else max_order + 1
            count_no_label = np.count_nonzero(np.array([self.tree.nodes[n][mode] for n in self.tree.nodes]) == 0)

        node = [n for n in self.tree.nodes if self.tree.nodes[n]['root']]
        if len(node) == 0:
            return
        node = node[0]

        neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])
        max_order = np.max(neighbor_orders)
        max_count = np.count_nonzero(neighbor_orders == max_order)
        self.tree.nodes[node][mode] = max_order if max_count == 1 and mode == 'HS' else max_order + 1

    def label_pressure_drop(self):
        for n in self.tree.edges:
            a, b = n[0], n[1]
            r = self.tree[a][b]['radius']

            l = np.linalg.norm(self.tree.nodes[a]['loc'] - self.tree.nodes[b]['loc']) * self.vsize
            self.tree[a][b]['length'] = l

            mu = self.mu

            self.tree[a][b]['resistance'] = 8 * mu * l / (np.pi * r ** 4)  # N s micrometer -5
            self.tree[a][b]['resistance_mm'] = 8 * mu * l * 1e15 / (np.pi * r ** 4)  # N s mm -5

            self.tree[a][b]['pressure_drop'] = self.tree[a][b]['resistance'] * self.tree[a][b]['flow']
            self.tree[a][b]['viscosity'] = mu

    def label_resistance(self):
        for n in self.tree.edges:
            a, b = n[0], n[1]
            r = self.tree[a][b]['radius']

            l = np.linalg.norm(self.tree.nodes[a]['loc'] - self.tree.nodes[b]['loc']) * self.vsize
            self.tree[a][b]['length'] = l

            mu = self.mu

            self.tree[a][b]['resistance'] = 8 * mu * l / (np.pi * r ** 4)  # N s micrometer -5
            self.tree[a][b]['resistance_mm'] = 8 * mu * l * 1e15 / (np.pi * r ** 4)  # N s mm -5

    def label_wss(self):
        for n in self.tree.edges:
            a, b = n[0], n[1]
            r = self.tree[a][b]['radius']

            l = np.linalg.norm(self.tree.nodes[a]['loc'] - self.tree.nodes[b]['loc']) * self.vsize
            self.tree[a][b]['length'] = l

            mu = self.mu

            self.tree[a][b]['wall_shear_stress'] = 4 * mu * self.tree[a][b]['flow'] / (np.pi * r ** 3)
            self.tree[a][b]['wall_shear_stress_Pa'] = self.tree[a][b]['wall_shear_stress'] * 1e12
            self.tree[a][b]['wall_shear_rate'] = 4 * self.tree[a][b]['flow'] / (np.pi * r ** 3)

    def label_pressure_from_root(self, node=None, root_pressure=0.0):

        if node is None:
            node = [n for n in self.tree.nodes if self.tree.nodes[n]['root']][0]
            self.tree.nodes[node]['pressure'] = root_pressure
            self.tree.nodes[node]['pressure_mmhg'] = self.tree.nodes[node]['pressure'] * 1e12 / 133.322

            children = np.array(list(self.tree.neighbors(node)))

        else:
            neighbors = np.array(list(self.tree.neighbors(node)))
            neighbor_orders = np.array([self.tree.nodes[n]['level'] for n in neighbors])
            root = -1 if -1 in neighbor_orders else np.max(neighbor_orders)
            children = np.array([neighbors[i] for i in range(len(neighbors)) if neighbor_orders[i] != root])
            root = neighbors[np.where(neighbor_orders == root)[0][0]]
            self.tree.nodes[node]['pressure'] = \
                self.tree.nodes[root]['pressure'] - self.tree[root][node]['pressure_drop']

            self.tree.nodes[node]['pressure_mmhg'] = self.tree.nodes[node]['pressure'] * 1e12 / 133.322

        for i in children:
            self.label_pressure_from_root(i)

    def label_pressure_from_root_di(self, node=None, root_pressure=0.0):

        if node is None:
            node = [n for n in self.tree.nodes if self.tree.nodes[n]['root']][0]
            self.tree.nodes[node]['pressure'] = root_pressure
            self.tree.nodes[node]['pressure_mmhg'] = self.tree.nodes[node]['pressure'] * 1e12 / 133.322

            children = np.array(list(self.tree.successors(node)))

        else:
            children = np.array(list(self.tree.successors(node)))
            root = list(self.tree.predecessors(node))[0]
            self.tree.nodes[node]['pressure'] = \
                self.tree.nodes[root]['pressure'] - self.tree[root][node]['pressure_drop']

            self.tree.nodes[node]['pressure_mmhg'] = self.tree.nodes[node]['pressure'] * 1e12 / 133.322

        for i in children:
            self.label_pressure_from_root_di(i)


    def save_terminal_only(self, file='sanity_check3.vtk'):

        tree_copy = copy.deepcopy(self.tree)

        all_terminals = [i for i in self.tree.nodes if len(list(self.tree.successors(i))) == 0]
        all_terminal_edges = np.array([(list(self.tree.predecessors(i))[0], i) for i in all_terminals])

        nodes_to_remove = np.setdiff1d(list(self.tree.nodes), np.reshape(all_terminal_edges, -1))

        all_edges = np.array(self.tree.edges)
        remaining_edges = (all_edges[:, None] == all_terminal_edges).all(-1).any(-1)
        edge_to_remove = all_edges[~remaining_edges]

        tree_copy.remove_nodes_from(nodes_to_remove)
        tree_copy.remove_edges_from(edge_to_remove)

        node_list = list(tree_copy.nodes)
        level_idices = np.argsort(node_list)
        level_map = {}
        for idx, i in enumerate(level_idices):
            level_map[node_list[i]] = idx

        tree_copy = nx.relabel_nodes(tree_copy, level_map)

        nodes_inds = tree_copy.nodes
        indices = sorted(nodes_inds)


        points = [nodes_inds[i]['loc'] for i in indices]
        points = np.array(points)


        edges_inds = tree_copy.edges
        edges_inds = np.array(edges_inds)

        if np.max(edges_inds) >= len(nodes_inds):
            map = dict(zip(indices, np.arange(len(indices))))
            edges_inds_new = np.vectorize(map.get)(edges_inds)
        else:
            edges_inds_new = edges_inds

        lines = np.hstack([np.ones((len(edges_inds_new), 1), dtype=np.uint8) * 2, edges_inds_new])
        lines = np.hstack(lines)
        mesh = pyvista.PolyData(points,
                                lines=lines
                                )

        for key in tree_copy[edges_inds[0][0]][edges_inds[0][1]]:
            edge_feature = [tree_copy[n1][n2][key] for n1, n2 in edges_inds]
            edge_feature = np.array(edge_feature)

            assert len(edge_feature) > 0 and len(edge_feature) == mesh.n_lines
            mesh.cell_data[key] = edge_feature

        for key in tree_copy.nodes[indices[0]]:
            if key in ('loc', 'fixed', 'root'):
                continue
            node_feature = [tree_copy.nodes[i][key] for i in indices]
            assert len(node_feature) > 0 and len(node_feature) == mesh.n_points
            mesh.point_data[key] = np.array(node_feature)

        mesh.save(file)

    def label_total_resistance(self, node=None):

        if node is None:
            node = [n for n in self.tree.nodes if self.tree.nodes[n]['root']][0]

        children = np.array(list(self.tree.successors(node)))
        if len(children) == 0:
            return 0

        eq = 1 / np.sum([1 / self.tree[node][i]['resistance'] for i in children])


        return eq + np.sum([self.label_total_resistance(i) for i in children])

    def reorder_nodes(self):

        all_nodes = list(self.tree.nodes)

        level_map = {all_nodes[i]: i for i in range(len(all_nodes))}

        self.tree = nx.relabel_nodes(self.tree, level_map)

    def reorder_removed_leaves(self, n):
        """
        Reorder nodes by level.
        """

        level_map = {}
        j = 0
        for i in self.tree.nodes:

            while j < len(n) and i > n[j]:
                j += 1

            level_map[i] = i - j

        assert len(level_map) == len(self.tree.nodes)

        self.tree = nx.relabel_nodes(self.tree, level_map)



    def Kirchoff_law_PC(self, P_in=100 * 133.322e-6, P_out=0, Q0=None, ratio=None):
        n_nodes = len(self.tree.nodes)

        assert np.max(self.tree.nodes) == n_nodes - 1

        all_terminals = [i for i in self.tree.nodes if len(list(self.tree.successors(i))) == 0]
        all_terminal_edges = [(list(self.tree.predecessors(i))[0], i) for i in all_terminals]

        A = np.zeros((n_nodes, n_nodes))
        b = np.zeros(n_nodes)

        R_e = 0.209 * 60 * 133.322 * 1e-3  # Ns mm-5 # around 1.67
        R_PC = 0.0702 * 60 * 133.322 * 1e-3  # Ns mm-5 around 0.56
        # ratio around 0.7
        for i in range(n_nodes):
            if len(list(self.tree.predecessors(i))) == 0:
                A[i, i] = 1
                b[i] = P_in
            elif len(list(self.tree.successors(i))) == 0:
                A[i, i] = 1
                b[i] = P_out

            else:
                assert len(list(self.tree.predecessors(i))) == 1
                n_in = list(self.tree.predecessors(i))[0]
                # R_in = self.tree[n_in][i]['resistance']         # N s mm -5
                R_in = self.tree[n_in][i]['resistance_mm']
                A[i, n_in] = - 1 / R_in
                coefficient_cur = 1 / R_in

                for n_out in self.tree.successors(i):
                    # R_out = self.tree[i][n_out]['resistance']

                    if len(list(self.tree.successors(n_out))) == 0:
                        R_out = self.tree[i][n_out]['resistance_mm'] + R_e + R_PC
                    else:
                        R_out = self.tree[i][n_out]['resistance_mm']
                    coefficient_cur += 1 / R_out
                    A[i, n_out] = - 1 / R_out

                A[i, i] = coefficient_cur



        res = np.linalg.solve(A, b)  # micro-meter3 s-1 = 1e-6 nl/s = 6e-5 nl/min

        for i in range(n_nodes):
            self.tree.nodes[i]['pressure_from_Kirchhoff'] = res[i]  #
            # self.tree.nodes[i]['pressure_from_Kirchhoff_mmHg'] = res[i]/(133.322e-12)
            self.tree.nodes[i]['pressure_from_Kirchhoff_mmHg'] = res[i] / (133.322e-6)

        all_terminal_flow = np.array([(self.tree.nodes[i]['pressure_from_Kirchhoff'] -
                                       self.tree.nodes[j]['pressure_from_Kirchhoff']) /
                                      (self.tree[i][j]['resistance_mm'] + R_e + R_PC) for (i, j) in all_terminal_edges])
        if Q0 is None and ratio is None:
            Q0 = all_terminal_flow / 6
        else:
            if Q0 is not None:
                Q0 = Q0 / 60 * 1e-3  # nl/min to mm3/s
            else:
                Q0 = all_terminal_flow * ratio

        for q0, qt, (i, j) in zip(Q0, all_terminal_flow, all_terminal_edges):
            Q_aa = qt + q0 * R_e / (R_e + self.tree[i][j]['resistance_mm'] + R_PC)
            P1 = self.tree.nodes[i]['pressure_from_Kirchhoff']
            P2 = self.tree.nodes[j]['pressure_from_Kirchhoff']
            assert np.isclose(P2, 0)
            AA_pressure = P1 - self.tree[i][j]['resistance_mm'] * Q_aa

            self.tree.nodes[j]['pressure_from_Kirchhoff'] = AA_pressure
            self.tree.nodes[j]['pressure_from_Kirchhoff_mmHg'] = AA_pressure / (133.322e-6)
            self.tree[i][j]['flow_from_Kirchhoff'] = Q_aa
            # N mm-2/(N s mm -5) = mm3/s
            self.tree[i][j]['flow_from_Kirchhoff_nl/s'] = Q_aa * 1e3
            # mm3 / s

        all_terminal_flow = [self.tree[i][j]['flow_from_Kirchhoff'] for (i, j) in all_terminal_edges]

        self.rescale_flow_Kirchhoff()
        root = [i for i in self.tree.nodes if len(list(self.tree.predecessors(i))) == 0]
        assert len(root) == 1
        root = root[0]
        root_edge = [root, list(self.tree.successors(root))[0]]
        in_flow = self.tree[root_edge[0]][root_edge[1]]['flow_from_Kirchhoff']  # mm3/s
        in_flow = in_flow / 1e3 * 60  # ml/min
        # print(f'{in_flow = } after Kirchhoff')

        # self.save('kirchhoff_pc.vtk')


    def rescale_flow_Kirchhoff(self, mode='level'):

        for n in self.tree.nodes:
            self.tree.nodes[n][mode] = 1 if self.tree.degree[n] <= 1 else 0

            if self.tree.nodes[n]['root']:
                assert len(list(self.tree.predecessors(n))) == 0
                self.tree.nodes[n][mode] = -1

        count_no_label = len(self.tree.nodes)

        while count_no_label != 0:
            for node in self.tree.nodes:
                if self.tree.nodes[node][mode] != 0 or len(list(self.tree.neighbors(node))) == 0:
                    continue

                neighbor_orders = np.array([self.tree.nodes[n][mode] for n in self.tree.neighbors(node)])

                root = list(self.tree.predecessors(node))[0]

                if np.count_nonzero(neighbor_orders == 0) >= 1:
                    continue

                max_order = np.max(neighbor_orders)
                self.tree.nodes[node][mode] = max_order + 1

                self.tree[root][node]['flow_from_Kirchhoff'] = np.sum(
                    [self.tree[node][n]['flow_from_Kirchhoff'] for n in self.tree.neighbors(node)])

                self.tree[root][node]['flow_from_Kirchhoff_nl/s'] = np.sum(
                    [self.tree[node][n]['flow_from_Kirchhoff_nl/s'] for n in self.tree.neighbors(node)])

            count_no_label = np.count_nonzero(np.array([self.tree.nodes[n][mode] for n in self.tree.nodes]) == 0)



    def find_terminal_all_neighbors(self, node):
        """
        Update order when no further operations will be applied.
        """
        parent = list(self.tree.predecessors(node))[0]
        all_children = list(self.tree.successors(parent))
        all_neighbors = [i for i in all_children if i != node]
        return np.array(all_neighbors)


## 7. Sparse full-tree Kirchhoff solver

In [ ]:
def kirchhoff_sparse(self, P_in=100 * 133.322e-6, P_out=0, Q0=None, ratio=None):
    """Equation-identical sparse form of the released Kirchhoff solve."""
    n_nodes = len(self.tree.nodes)
    assert np.max(self.tree.nodes) == n_nodes - 1
    terminals = [i for i in self.tree.nodes if self.tree.out_degree(i) == 0]
    terminal_edges = [(next(self.tree.predecessors(i)), i) for i in terminals]
    matrix = lil_matrix((n_nodes, n_nodes), dtype=float)
    rhs = np.zeros(n_nodes)
    r_e = 0.209 * 60 * 133.322 * 1e-3
    r_pc = 0.0702 * 60 * 133.322 * 1e-3

    for node in range(n_nodes):
        parents = list(self.tree.predecessors(node))
        children = list(self.tree.successors(node))
        if not parents:
            matrix[node, node] = 1.0
            rhs[node] = P_in
        elif not children:
            matrix[node, node] = 1.0
            rhs[node] = P_out
        else:
            assert len(parents) == 1
            parent = parents[0]
            r_in = self.tree[parent][node]["resistance_mm"]
            matrix[node, parent] = -1.0 / r_in
            diagonal = 1.0 / r_in
            for child in children:
                r_out = self.tree[node][child]["resistance_mm"]
                if self.tree.out_degree(child) == 0:
                    r_out += r_e + r_pc
                diagonal += 1.0 / r_out
                matrix[node, child] = -1.0 / r_out
            matrix[node, node] = diagonal

    pressure = spsolve(matrix.tocsr(), rhs)
    for node in range(n_nodes):
        self.tree.nodes[node]["pressure_from_Kirchhoff"] = pressure[node]
        self.tree.nodes[node]["pressure_from_Kirchhoff_mmHg"] = pressure[node] / 133.322e-6

    total_flow = np.array([
        (self.tree.nodes[i]["pressure_from_Kirchhoff"] - self.tree.nodes[j]["pressure_from_Kirchhoff"])
        / (self.tree[i][j]["resistance_mm"] + r_e + r_pc)
        for i, j in terminal_edges
    ])
    if Q0 is None and ratio is None:
        filtration = total_flow / 6.0
    elif Q0 is not None:
        filtration = np.asarray(Q0) / 60.0 * 1e-3
    else:
        filtration = total_flow * np.asarray(ratio)

    for q0, qt, (i, j) in zip(filtration, total_flow, terminal_edges):
        q_aa = qt + q0 * r_e / (r_e + self.tree[i][j]["resistance_mm"] + r_pc)
        p1 = self.tree.nodes[i]["pressure_from_Kirchhoff"]
        assert np.isclose(self.tree.nodes[j]["pressure_from_Kirchhoff"], 0)
        p_aa = p1 - self.tree[i][j]["resistance_mm"] * q_aa
        self.tree.nodes[j]["pressure_from_Kirchhoff"] = p_aa
        self.tree.nodes[j]["pressure_from_Kirchhoff_mmHg"] = p_aa / 133.322e-6
        self.tree[i][j]["flow_from_Kirchhoff"] = q_aa
        self.tree[i][j]["flow_from_Kirchhoff_nl/s"] = q_aa * 1e3
    self.rescale_flow_Kirchhoff()


# Same equations and coefficients as VtkNetworkAnalysis.Kirchoff_law_PC,
# but sparse storage avoids a prohibitively large dense matrix.
VtkNetworkAnalysis.Kirchoff_law_PC = kirchhoff_sparse


## 8. Released diagnostic plotting functions

In [ ]:
rcParams.update({'figure.autolayout': True})

def plot_save_results_before_iteration(df, _iter, path, P_in):

    all_terminal_radius, all_terminal_pressures, all_terminal_flow, T0_all, all_terminal_resistance = df.T

    if _iter == -1:

        plt.figure()
        plt.scatter(all_terminal_pressures, all_terminal_flow)
        plt.title(f'terminal pressure vs flow')
        plt.savefig(os.path.join(path, f"terminal pressure vs flow_0.png"), format="png", bbox_inches="tight")
        plt.clf()
        plt.figure()
        plt.hist(T0_all, 30)
        plt.title(f'T0 mean = {np.mean(T0_all)}, std = {np.std(T0_all)} dyn/cm')
        plt.savefig(os.path.join(path, f"T0_{0}.png"), format="png", bbox_inches="tight")
        plt.clf()
        print(f'T0 = {np.mean(T0_all):.4f} at P0 = {int(P_in / 133.322e-6)} at {_iter = }  dyn/cm')

        df = pd.DataFrame(df, columns=['radius', 'pressure', 'flow', 'T', 'resistance'])
        df.to_csv(os.path.join(path, 'before_iter.csv'), sep=',', index=False)
        QA0 = all_terminal_flow / 2 * 1e-6 * 60
        plt.figure()
        plt.hist(QA0, 30)
        plt.title(f'mean terminal_flow = {np.mean(QA0)}, std = {np.std(QA0)} nl/min')
        plt.savefig(os.path.join(path, f"QA_0.png"), format="png", bbox_inches="tight")
        plt.clf()

        Q_T00 = QA0 / 3

        plt.figure()
        plt.hist(Q_T00, 30)
        plt.title(f'mean QT0 = {np.mean(Q_T00)}, std = {np.std(Q_T00)} nl/min')
        plt.savefig(os.path.join(path, f"QT0_0.png"), format="png", bbox_inches="tight")
        plt.clf()

    plt.figure()
    plt.scatter(all_terminal_pressures, T0_all)
    plt.xlabel('terminal pressure')
    plt.ylabel('Tension')
    plt.savefig(os.path.join(path, f"tension_vs_pressure{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    all_terminal_flow1 = all_terminal_flow * 1e-6  # nl/s
    plt.figure()
    plt.hist(all_terminal_flow1, 30)
    plt.title(f'sum = {np.sum(all_terminal_flow1) / 1e6 * 60  :.4f} ml/min, '
              f'mean terminal_flow = {np.mean(all_terminal_flow1 * 60)}, std = {np.std(all_terminal_flow1 * 60)} nl/min')
    plt.savefig(os.path.join(path, f"flow_distribution_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_resistance, 30)
    plt.title(f'mean = {np.mean(all_terminal_resistance):.4f}')
    plt.savefig(os.path.join(path, f"resistance_distribution_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_radius, all_terminal_pressures)
    plt.xlabel('radius')
    plt.ylabel('pressure')
    plt.title('all_terminal_pressure vs all_terminal_radius')
    plt.savefig(os.path.join(path, f"pressure_vs radius_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_radius, all_terminal_flow)
    plt.xlabel('radius')
    plt.ylabel('flow')
    plt.title('all_terminal_flow vs all_terminal_radius')
    plt.savefig(os.path.join(path, f"flow_vs_radius{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_pressures, 30)
    plt.title(f'mean = {np.mean(all_terminal_pressures):.4f}, std = {np.std(all_terminal_pressures)}')
    plt.savefig(os.path.join(path, f"pressure_distribution_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_flow, all_terminal_pressures)
    plt.title('all_terminal_pressure vs all_terminal_flow')
    plt.savefig(os.path.join(path, f"pressure vs flow_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_radius, 30)
    plt.title(f'mean = {np.mean(all_terminal_radius):.4f}, std = {np.std(all_terminal_radius)}')
    plt.savefig(os.path.join(path, f"radius_distribution_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()


def plot_save_results(df, in_flow, _iter, path):
    (all_terminal_radius, all_terminal_pressures, all_terminal_flow, all_terminal_T, all_terminal_resistance,
         all_terminal_filtration, all_terminal_x_myo, all_terminal_x_tgf, all_terminal_Cs_md, all_terminal_T_e,
         all_terminal_T_m, all_terminal_q_d_end, all_terminal_P_end, all_terminal_P_md) = df.T
    df = pd.DataFrame(df, columns=['radius', 'pressure', 'flow', 'T',
                                   'resistance', 'filtration', 'x_myo', 'x_tgf',
                                   'Cs_md', 'T_e', 'T_m', 'q_end', 'P_end', 'P_md'])

    df.to_csv(os.path.join(path, f'iter_{_iter}.csv'), sep=',', index=False)

    plt.figure()
    plt.hist(all_terminal_q_d_end, 30)
    plt.title('all_terminal_q_d_end')
    plt.savefig(os.path.join(path,f"q_end_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_P_end, 30)
    plt.title('all_terminal_P_end')
    plt.savefig(os.path.join(path,f"all_terminal_P_end_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_P_md, 30)
    plt.title('all_terminal_P_md')
    plt.savefig(os.path.join(path, f"all_terminal_P_md_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_T, all_terminal_Cs_md)
    plt.title('Cs_md vs T')
    plt.xlabel('T')
    plt.ylabel('Cs_md')
    plt.savefig(os.path.join(path,f"Cs_md_vs_Tension{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()


    plt.figure()
    plt.scatter(all_terminal_T, all_terminal_x_myo)
    plt.title('x_myo vs T')
    plt.xlabel('T')
    plt.ylabel('x_myo')
    plt.savefig(os.path.join(path,f"x_myo vs T_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()


    plt.figure()
    plt.scatter(all_terminal_x_myo, all_terminal_Cs_md)
    plt.title('Cs_md vs x_myo')
    plt.xlabel('x_myo')
    plt.ylabel('Cs_md')
    plt.savefig(os.path.join(path,f"x_myo_vs_Cs_md{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_Cs_md, all_terminal_x_tgf)
    plt.title('x_tgf vs Cs_md')
    plt.xlabel('Cs_md')
    plt.ylabel('x_tgf')
    plt.savefig(os.path.join(path,f"x_tgf vs Cs_md{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_T_e, 30)
    plt.title('T_e')
    plt.savefig(os.path.join(path,f"T_e_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_T_m, 30)
    plt.title('T_m')
    plt.savefig(os.path.join(path,f"T_m_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_T_m, all_terminal_T_e)
    plt.title('Te vs Tm')
    plt.xlabel('Tm')
    plt.ylabel('Te')
    plt.savefig(os.path.join(path, f"Te vs Tm_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()


    plt.figure()
    plt.scatter(all_terminal_flow, all_terminal_Cs_md)
    plt.xlabel('all_terminal_flow')
    plt.ylabel('all_terminal_Cs_md')
    plt.title('all_terminal_Cs_md vs all_terminal_flow')
    plt.savefig(os.path.join(path, f"all_terminal_Cs_md vs all_terminal_flow_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_radius, all_terminal_Cs_md)
    plt.title('all_terminal_Cs_md vs all_terminal_radius')
    plt.xlabel('all_terminal_radius')
    plt.ylabel('all_terminal_Cs_md')
    plt.savefig(os.path.join(path, f"all_terminal_Cs_md vs all_terminal_radius_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.scatter(all_terminal_x_myo, all_terminal_x_tgf)
    plt.xlabel('x_myo')
    plt.ylabel('x_tgf')
    plt.savefig(os.path.join(path,f"tgf_vs_myo_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_x_tgf, 30)
    plt.title('tgf hist')
    plt.savefig(os.path.join(path,f"tgf_hist_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_x_myo, 30)
    plt.title('myo hist')
    plt.savefig(os.path.join(path, f"myo_hist_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    plt.figure()
    plt.hist(all_terminal_Cs_md, 30)
    plt.title(f'mean Cs_md = {np.mean(all_terminal_Cs_md):.4f}')
    plt.savefig(os.path.join(path,f"all_terminal_Cs_md_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()


    plt.figure()
    plt.hist(all_terminal_filtration, 30)
    plt.savefig(os.path.join(path,f"filtration_distribution_{_iter}.png"), format="png", bbox_inches="tight")
    plt.clf()

    total_filtration = np.sum(all_terminal_filtration) * 1e-6  # ml/min

    r_mean = np.mean(all_terminal_radius)
    Cs_md_mean = np.mean(all_terminal_Cs_md)
      # mm3/s
    # in_flow = in_flow / 1e3 * 60  # 7 ml min -1
    print(f'in_flow = {in_flow:.4f} ml/min, r = {r_mean:.4f} Cs_md = {Cs_md_mean:.4f}'
          f' total_filtration = {total_filtration:.4f} ml/s at {_iter = }')

    logging.warning(f'in_flow = {in_flow:.4f} ml/min, r = {r_mean:.4f} Cs_md = {Cs_md_mean:.4f}'
          f' total_filtration = {total_filtration:.4f} ml/s at {_iter = }')


    print(f'**************** iter {_iter} finished ************************')
    logging.warning(f'**************** iter {_iter} finished ************************')


In [ ]:
# Preserve the complete published plotting functions above, while allowing a
# lightweight run. AutoRegulation resolves these global names at runtime.
_full_plot_before = plot_save_results_before_iteration
_full_plot_after = plot_save_results

def plot_save_results_before_iteration(*args, **kwargs):
    if SAVE_DETAILED_ITERATION_PLOTS:
        return _full_plot_before(*args, **kwargs)

def plot_save_results(*args, **kwargs):
    if SAVE_DETAILED_ITERATION_PLOTS:
        return _full_plot_after(*args, **kwargs)


## 9. Coupled terminal-nephron calculation and outer autoregulation iteration

For every terminal, `simu` computes glomerular/tubular state and a target afferent radius. `AutoRegulation` updates all terminal radii, recomputes resistance, resolves the whole tree, and repeats until renal blood flow converges.

In [ ]:
# import multiprocessing
warnings.filterwarnings("error")

plt.rcParams.update({'figure.max_open_warning': 0})


def simu(Q, r_v, P_t_in, P_GC, type=0, only_myo=False):

    Q_A = 6 * 1e-5 * Q * (1 - Glomerular.H_A)  # micro-meter3 s-1 to nanolitter min-1
    P_v = (P_t_in + P_GC)/2

    if type == 0:
        func_glomerular_cur = func_glomerular
        AA_cur = AA_model

    elif type == 1:
        func_glomerular_cur = partial(func_glomerular,
                                  inter=True
                                  )
        AA_cur = partial(AA_model, type=type)
    elif type == 2:
        func_glomerular_cur = partial(func_glomerular,
                                  long=True
                                  )
        AA_cur = partial(AA_model, type=type)
    else:
        raise "this should never happen, nephron type needs to be within 3 types"

    try:
        root = fsolve(func_glomerular_cur, x0=10, args=(Q_A, P_GC))
        assert len(root) == 1
        P_0_final = root[0]
        P_0_final_again, Cs_md_final, Q_T0, Cs_desc_end, Q_T_desc_end, P_end, P_md = func_glomerular_cur(P_0_final, Q_A, P_GC=P_GC,
                                                                                            final=True)

    except NegativeFlowException as e:
        logging.info(f"FlowException in finding P0: with {Q_A = }  {P_GC = } Cs_md is set to be 0")
        print(f"FlowException in finding P0: with {Q_A = }  {P_GC = } Cs_md is set to be 0", end=', ')
        P_0_final, P_0_final_again, Cs_md_final, Q_T0, Cs_desc_end, Q_T_desc_end, P_end, P_md  = 0, 0, 0, 0, 0, 0, 0, 0
        print(e)

    except RuntimeWarning as e:
        print(f"RuntimeWarning in finding P0: with {Q_A = }  {P_GC = } no regulation happen here", end=', ')
        print(e)
        logging.warning(f"RuntimeWarning in finding P0: with {Q_A = }  {P_GC = } no regulation happen here")
        logging.warning(e)

        return r_v, 1/3 * Q_A, 0, 0, 0, 0, 0, 0, 1/3, 0, 0, 0, 0

    ratio = Q_T0/(Q_A*2)

    try:
        r_opt = so.brentq(AA_cur, a=1e-1, b=20, args=(Cs_md_final, P_v, False, only_myo))
        T1, x_myo, x_tgf, T_e, T_m = AA_cur(r_opt,  Cs_md_final, P_v, final=True, only_myo=only_myo)

    except RuntimeWarning as e:
        print(f"RuntimeWarning in finding r_new: with {Q_A = } {P_GC = } {Cs_md_final = } {P_v = }")
        print(e)
        logging.warning(e)

        T1, x_myo, x_tgf, T_e, T_m = AA_cur(r_v, Cs_md_final, P_v, final=True, only_myo=only_myo)
        
        return r_v, Q_T0, 0, 0, Cs_md_final, P_0_final, Cs_desc_end, Q_T_desc_end, ratio, T_e, T_m, P_end, P_md

    except Exception as e:
        print('This should never happen, other exception not RuntimeWarnings ', end=', ')
        logging.warning(f"Warning in finding r_new: with {Q_A = } {P_GC = } {Cs_md_final = } {P_v = }")
        print(e)
        sys.exit()

    logging.info(f'{P_0_final_again:.2f}, {Cs_md_final:.2f}, {r_opt:.2f}')


    return r_opt, Q_T0, x_myo, x_tgf, Cs_md_final, P_0_final, Cs_desc_end, Q_T_desc_end, ratio, T_e, T_m, P_end, P_md


class AutoRegulation:
    def __init__(self, P_in=100, lr=0.3, only_myo=False, num_iter=20, path=None, pop=True, relTol=1e-6):
        self.P_in = P_in * 133.322e-6
        self.lr = lr
        self.only_myo = only_myo
        self.num_iter = num_iter
        self.pt_file = None
        self.path = path
        self.pop = pop
        self.relTol = relTol

    def set_up_tree(self, root_loc=[588, 217, 650], t_flow=1.167e11 / 3e4, mu=3.6e-15, vspace=22.6, pt_file=None,
                    surface_name='surface.nii.gz'):

        self.surface_name = surface_name

        if pt_file is None:
            pt_file = os.path.join('data', f'kirchhoff_p_in = {self.P_in}.vtk')

        if not os.path.exists(pt_file):
            pt_file = 'final_tree.vtk'

        self.pt_file = pt_file

        vt = VtkNetworkAnalysis(pt_file, root_loc, vsize=vspace, mu=mu, t_flow=t_flow)
        vt.build()
        vt.to_directed()
        vt.tree = vt.di_tree

        vt.label_pressure_drop()
        vt.label_pressure_from_root_di(
            root_pressure=self.P_in * 1e-6
        )


        if 'kir' not in pt_file:
            vt.Kirchoff_law_PC(P_in=self.P_in, Q0=None)  # N/mm2
            vt.save(f'kirchhoff_p_in = {int(self.P_in/133.322e-6)}.vtk')

        if self.path is None:
            path = (f'path_new_fix_C_params_{int(self.P_in/133.322e-6)}, '
                    f'{self.lr = }_only_myo_{self.only_myo}_pop_{self.pop}')
        else:
            path = self.path

        if not os.path.exists(path):
            os.mkdir(path)
        logging.basicConfig(filename=os.path.join(path, 'app_tree_only.log'),
                            level=logging.WARNING,
                            filemode="w",
                            # format='%(asctime)s - %(levelname)s - %(message)s'
                            )


        print(f'P in = {int(self.P_in/133.322e-6)}')
        logging.warning(f'P in = {int(self.P_in/133.322e-6)}')
        logging.warning(f'{self.lr = }')
        print(f'{self.lr = }')

        all_terminals = [i for i in vt.tree.nodes if len(list(vt.tree.successors(i))) == 0]
        all_terminal_edges = [(list(vt.tree.predecessors(i))[0], i) for i in all_terminals]
        root = [i for i in vt.tree.nodes if len(list(vt.tree.predecessors(i))) == 0]
        assert len(root) == 1
        root = root[0]
        root_edge = [root, list(vt.tree.successors(root))[0]]
        all_terminal_flow = np.array([vt.tree[i][j]['flow'] for (i, j) in all_terminal_edges])
        assert np.all(all_terminal_flow == all_terminal_flow[0])
        # micro-meter3 s-1 # N s micro-meter-2

        all_terminal_pressures = np.array([vt.tree.nodes[i]['pressure_mmhg'] for i in all_terminals])
        all_terminal_pressures_in = np.array([vt.tree.nodes[i]['pressure_mmhg'] for (i, _) in all_terminal_edges])

        plt.figure()
        plt.hist(all_terminal_pressures, 30)
        plt.title(f'mean pressure = {np.mean(all_terminal_pressures)}, std = {np.std(all_terminal_pressures)}')
        plt.savefig(os.path.join(path, f"pressure_distribution_before_kirchhoff.png"), format="png", bbox_inches="tight")
        plt.clf()

        plt.figure()
        plt.hist(all_terminal_flow, 30)
        plt.title(f'mean terminal_flow = {np.mean(all_terminal_flow)}, std = {np.std(all_terminal_flow)}')
        plt.savefig(os.path.join(path, f"flow_distribution_before_kirchhoff.png"), format="png", bbox_inches="tight")
        plt.clf()

        logging.info('P_0_final, Cs_md_final, r_opt')

        in_flow = vt.tree[root_edge[0]][root_edge[1]]['flow_from_Kirchhoff']  # mm3/s
        in_flow = in_flow / 1e3 * 60

        for i in vt.tree.nodes:
            vt.tree.nodes[i]['pressure'] = vt.tree.nodes[i]['pressure_from_Kirchhoff'] * 1e-6  #
            vt.tree.nodes[i]['pressure_mmHg'] = vt.tree.nodes[i]['pressure_from_Kirchhoff_mmHg']

        print(f'in_flow = {in_flow:.4f} ml/min before iterations')
        logging.warning(f'in_flow = {in_flow:.4f} ml/min before iterations')

        all_terminal_type = np.zeros(len(all_terminals))
        if self.pop:
            surface = nib.load(surface_name).get_fdata()
            surface = np.array(np.where(surface)).T
            all_terminal_pos = np.array([vt.tree.nodes[i]['loc'] for i in all_terminals])

            sb = scipy.spatial.KDTree(surface)
            all_terminal_dist_to_surface = sb.query(all_terminal_pos)
            all_terminal_dist_to_surface = all_terminal_dist_to_surface[0]
            all_terminal_dist_to_surface *= 22.6

            quant_1 = np.quantile(all_terminal_dist_to_surface, 0.5)
            quant_2 = np.quantile(all_terminal_dist_to_surface, 0.7)


            all_terminal_type[np.logical_and(all_terminal_dist_to_surface > quant_1,
                                             all_terminal_dist_to_surface < quant_2)] = 1
            all_terminal_type[all_terminal_dist_to_surface > quant_2] = 2


        self.all_terminals = all_terminals
        self.all_terminal_edges = all_terminal_edges
        self.root = root
        self.root_edge = root_edge
        self.path = path
        self.logging = logging
        self.vt = vt
        self.all_terminal_type = all_terminal_type

    def re_label_tree(self):

        all_terminals = [i for i in self.vt.tree.nodes if len(list(self.vt.tree.successors(i))) == 0]
        all_terminal_edges = [(list(self.vt.tree.predecessors(i))[0], i) for i in all_terminals]
        root = [i for i in self.vt.tree.nodes if len(list(self.vt.tree.predecessors(i))) == 0]
        assert len(root) == 1
        root = root[0]
        root_edge = [root, list(self.vt.tree.successors(root))[0]]

        self.all_terminals = all_terminals
        self.all_terminal_edges = all_terminal_edges
        self.root = root
        self.root_edge = root_edge

    def auto_reg(self):

        in_flow_before = self.vt.tree[self.root_edge[0]][self.root_edge[1]]['flow_from_Kirchhoff']  # mm3/s
        in_flow_before = in_flow_before / 1e3 * 60

        in_flow_cur = in_flow_before

        for i in self.vt.tree.nodes:
            self.vt.tree.nodes[i]['pressure'] = self.vt.tree.nodes[i]['pressure_from_Kirchhoff'] * 1e-6  #
            self.vt.tree.nodes[i]['pressure_mmHg'] = self.vt.tree.nodes[i]['pressure_from_Kirchhoff_mmHg']

        self.logging.warning(f'in_flow = {in_flow_before:.4f} ml/min before iterations')

        for _iter in range(self.num_iter):
            all_terminal_radius = np.array([self.vt.tree[i][j]['radius'] for (i, j) in self.all_terminal_edges])
            all_terminal_pressures = np.array([self.vt.tree.nodes[i]['pressure_from_Kirchhoff_mmHg'] for i in self.all_terminals])
            all_terminal_pressures_in = np.array(
                [self.vt.tree.nodes[i]['pressure_from_Kirchhoff_mmHg'] for (i, _) in self.all_terminal_edges])
            all_terminal_flow = np.array([self.vt.tree[i][j]['flow_from_Kirchhoff'] for (i, j) in self.all_terminal_edges]) * 1e9
            # N mm-2/(N s mm -5) = mm3/s = 10^9  micro-meter3/s
            all_terminal_resistance = np.array([self.vt.tree[i][j]['resistance_mm'] for (i, j) in self.all_terminal_edges])

            P_v_all = (all_terminal_pressures_in + all_terminal_pressures) / 2
            T0_all = P_v_all * all_terminal_radius / (1e3 / 133.32)

            logging.warning(f'T0 = {np.mean(T0_all):.4f} at P0 = {int(self.P_in/133.322e-6)} at {_iter = }')
            df = np.array([all_terminal_radius, all_terminal_pressures, all_terminal_flow, T0_all,
                           all_terminal_resistance]).T
            if _iter == 0:
                plot_save_results_before_iteration(df, -1, self.path, self.P_in)

            else:
                plot_save_results_before_iteration(df, _iter, self.path, self.P_in)

            radius_before = all_terminal_radius

            #
            # res = [simu(all_terminal_flow[i], all_terminal_radius[i], all_terminal_pressures_in[i],
            #             all_terminal_pressures[i], self.all_terminal_type[i],
            #             only_myo=self.only_myo) for i in range(len(all_terminal_flow))]



            inputs = (all_terminal_flow, all_terminal_radius, all_terminal_pressures_in,
                         all_terminal_pressures, self.all_terminal_type,
                      np.array([self.only_myo] * len(all_terminal_flow)))

            with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
                res = executor.map(simu, *inputs)

            executor = None

            res = np.array(list(res))

            res[:, 0] = radius_before + (res[:, 0] - radius_before) * self.lr

            for (r, q_t_0, x_myo_t, x_tgf_t, cs_md, p0, cs_d_end, qt_d_end, ratio, t_e, t_m, P_end, P_md, *_), (parent, i) in (
                    zip(res, self.all_terminal_edges)):
                self.vt.tree[parent][i]['radius'] = r
                self.vt.tree[parent][i]['filtration_rate'] = q_t_0
                self.vt.tree[parent][i]['x_myo'] = x_myo_t
                self.vt.tree[parent][i]['x_tgf'] = x_tgf_t
                self.vt.tree[parent][i]['Cs_md'] = cs_md
                self.vt.tree[parent][i]['P0'] = p0
                self.vt.tree[parent][i]['cs_d_end'] = cs_d_end
                self.vt.tree[parent][i]['qt_d_end'] = qt_d_end
                self.vt.tree[parent][i]['ratio'] = ratio
                self.vt.tree[parent][i]['T_e'] = t_e
                self.vt.tree[parent][i]['T_m'] = t_m
                self.vt.tree[parent][i]['P_end'] = P_end
                self.vt.tree[parent][i]['P_md'] = P_md

            self.vt.label_resistance()

            all_terminal_ratio = np.array([self.vt.tree[i][j]['ratio'] for (i, j) in self.all_terminal_edges])

            self.vt.Kirchoff_law_PC(P_in=self.P_in, ratio=all_terminal_ratio)  # N/mm2

            in_flow = self.vt.tree[self.root_edge[0]][self.root_edge[1]]['flow_from_Kirchhoff']  # mm3/s
            in_flow = in_flow / 1e3 * 60  # 7 ml min -1

            print(f'in_flow = {in_flow:.4f} ml/min at {_iter = }')
            logging.warning(f'in_flow = {in_flow:.4f} ml/min at {_iter = }')


            all_terminal_filtration = np.array([self.vt.tree[i][j]['filtration_rate'] for (i, j) in self.all_terminal_edges])
            all_terminal_x_myo = np.array([self.vt.tree[i][j]['x_myo'] for (i, j) in self.all_terminal_edges])
            all_terminal_x_tgf = np.array([self.vt.tree[i][j]['x_tgf'] for (i, j) in self.all_terminal_edges])
            all_terminal_pt0 = np.array([self.vt.tree[i][j]['P0'] for (i, j) in self.all_terminal_edges])
            all_terminal_Cs_md = np.array([self.vt.tree[i][j]['Cs_md'] for (i, j) in self.all_terminal_edges])
            all_terminal_Cs_d_end = np.array([self.vt.tree[i][j]['cs_d_end'] for (i, j) in self.all_terminal_edges])
            all_terminal_q_d_end = np.array([self.vt.tree[i][j]['qt_d_end'] for (i, j) in self.all_terminal_edges])
            all_terminal_radius = np.array([self.vt.tree[i][j]['radius'] for (i, j) in self.all_terminal_edges])
            all_terminal_T_e = np.array([self.vt.tree[i][j]['T_e'] for (i, j) in self.all_terminal_edges])
            all_terminal_T_m = np.array([self.vt.tree[i][j]['T_m'] for (i, j) in self.all_terminal_edges])
            all_terminal_T = all_terminal_T_e + all_terminal_T_m

            all_terminal_P_end =np.array([self.vt.tree[i][j]['P_end'] for (i, j) in self.all_terminal_edges])
            all_terminal_P_md =np.array([self.vt.tree[i][j]['P_md'] for (i, j) in self.all_terminal_edges])

            df = np.array([all_terminal_radius, all_terminal_pressures, all_terminal_flow, all_terminal_T,
                           all_terminal_resistance, all_terminal_filtration, all_terminal_x_myo, all_terminal_x_tgf,
                           all_terminal_Cs_md, all_terminal_T_e, all_terminal_T_m,
                           all_terminal_q_d_end, all_terminal_P_end, all_terminal_P_md]).T

            plot_save_results(df, in_flow, _iter, self.path)

            cur_flow_change = abs(in_flow - in_flow_cur) / (abs(in_flow) + 1e-8)

            print(f'relative flow change at {_iter = } cur_flow_change = {cur_flow_change:.5f}')
            logging.warning(f'relative flow change at {_iter = } cur_flow_change = {cur_flow_change:.5f}')

            if cur_flow_change < self.relTol:
                print(f'converged at {_iter = }')
                logging.warning(f'converged at {_iter = }')
                break

            else:
                in_flow_cur = in_flow
        # Converged

        return in_flow_before, in_flow


In [ ]:
GSA_PARAMETER_TABLE_PREVIEW = pd.read_excel(PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET)
assert len(GSA_PARAMETER_TABLE_PREVIEW) == 44
GSA_PARAMETER_TABLE_PREVIEW.head()

In [ ]:
PARAMETER_TARGETS = [
    "P_IN_MMHG",
    "Glomerular.C_A",
    "Glomerular.H_A",
    "Glomerular.Kf",
    "Glomerular.R_E",
    "Glomerular.R_PC",
    "Glomerular.a",
    "Glomerular.b",
    "Glomerular.L",
    "Tubular.Km",
    "Tubular.Lv",
    "Tubular.ns",
    "Tubular.Ls",
    "Tubular.V_max1",
    "Tubular.V_max2",
    "Tubular.alpha",
    "Tubular.beta",
    "Tubular.theta",
    "Tubular.keppa",
    "Tubular.gamma",
    "Tubular.mu",
    "Tubular.r_proximal",
    "Tubular.r_loop",
    "Tubular.Ls_long",
    "Tubular.Ls_inter",
    "AA.sigma_e",
    "AA.k_e",
    "AA.r_e",
    "AA.k_m",
    "AA.r_m",
    "AA.sigma_m",
    "Myo.lam",
    "Myo.T0",
    "Myo.G",
    "TGF.C_half",
    "TGF.k",
    "TGF.ita_max",
    "TGF.phi",
    "TGF_INTER.k",
    "TGF_INTER.ita_max",
    "TGF_INTER.phi",
    "TGF_LONG.k",
    "TGF_LONG.ita_max",
    "TGF_LONG.phi",
]

PARAMETER_CLASSES = {
    "Glomerular": Glomerular,
    "Tubular": Tubular,
    "AA": AA,
    "Myo": Myo,
    "TGF": TGF,
    "TGF_INTER": TGF_INTER,
    "TGF_LONG": TGF_LONG,
}

FIXED_FORWARD_SETTINGS = {
    "BLOOD_VISCOSITY": 3.6e-15,
    "ROOT_LOCATION": [588, 217, 650],
    "VOXEL_SIZE_UM": 22.6,
    "USE_AUTOREGULATION": True,
    "ONLY_MYOGENIC": False,
    "USE_NEPHRON_POPULATIONS": True,
}


def _evaluate_baseline_value(name):
    if name == "P_IN_MMHG":
        return float(P_IN_MMHG)
    cls_name, attr = name.split(".", 1)
    return float(getattr(PARAMETER_CLASSES[cls_name], attr))


BASELINE_PARAMETER_VALUES = {name: _evaluate_baseline_value(name) for name in PARAMETER_TARGETS}


def load_parameter_table(path=PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET):
    df = pd.read_excel(path, sheet_name=sheet_name)
    required = {
        "parameter_name",
        "nominal_value",
        "final_lower",
        "final_upper",
        "unit",
        "xu_functional_module",
        "include_in_oat",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Parameter table is missing required columns: {missing}")
    df = df.copy()
    if "input_index" not in df.columns:
        df.insert(0, "input_index", np.arange(1, len(df) + 1))
    df["input_index"] = df["input_index"].astype(int)
    df["group_id"] = df["parameter_name"]
    df["members"] = df["parameter_name"]
    df["baseline"] = pd.to_numeric(df["nominal_value"], errors="raise")
    df["lower_bound"] = pd.to_numeric(df["final_lower"], errors="raise")
    df["upper_bound"] = pd.to_numeric(df["final_upper"], errors="raise")
    if len(df) != 44:
        raise ValueError(f"Expected exactly 44 Xu parameters, found {len(df)}")
    if df["parameter_name"].duplicated().any():
        dup = df.loc[df["parameter_name"].duplicated(), "parameter_name"].tolist()
        raise ValueError(f"Duplicate parameter names: {dup[:5]}")
    missing_targets = sorted(set(PARAMETER_TARGETS) - set(df["parameter_name"]))
    extra_targets = sorted(set(df["parameter_name"]) - set(PARAMETER_TARGETS))
    if missing_targets or extra_targets:
        raise ValueError(f"Parameter mismatch. Missing={missing_targets}; extra={extra_targets}")
    return df.sort_values("input_index").reset_index(drop=True)


def get_parameter_value(name):
    if name == "P_IN_MMHG":
        return float(P_IN_MMHG)
    cls_name, attr = name.split(".", 1)
    return float(getattr(PARAMETER_CLASSES[cls_name], attr))


def set_parameter_value(name, value):
    value = float(value)
    if name == "P_IN_MMHG":
        globals()["P_IN_MMHG"] = value
        return
    cls_name, attr = name.split(".", 1)
    setattr(PARAMETER_CLASSES[cls_name], attr, value)


def validate_parameter_table_against_model(parameter_table=None, atol=1e-12, rtol=1e-9):
    if parameter_table is None:
        parameter_table = load_parameter_table()
    rows = []
    for _, row in parameter_table.iterrows():
        name = row["parameter_name"]
        expected = float(row["baseline"])
        actual = get_parameter_value(name)
        ok = np.isclose(actual, expected, atol=atol, rtol=rtol)
        rows.append({
            "input_index": int(row["input_index"]),
            "parameter_name": name,
            "baseline_expected": expected,
            "model_baseline": actual,
            "baseline_matches_model": bool(ok),
            "recognized_by_model": name in PARAMETER_TARGETS,
        })
    return pd.DataFrame(rows)


class XuParameterContext:
    def __init__(self, parameter_values=None):
        self.parameter_values = parameter_values or {}
        self.previous = {}

    def __enter__(self):
        names = set(self.parameter_values) | set(FIXED_FORWARD_SETTINGS)
        for name in names:
            if name in PARAMETER_TARGETS:
                self.previous[name] = get_parameter_value(name)
            elif name in globals():
                self.previous[name] = deepcopy(globals()[name])
        for name, value in FIXED_FORWARD_SETTINGS.items():
            if name in globals():
                globals()[name] = deepcopy(value)
        for name, value in self.parameter_values.items():
            if name not in PARAMETER_TARGETS:
                raise KeyError(f"Unknown Xu 44 parameter: {name}")
            set_parameter_value(name, value)
        refresh_runtime_paths()
        return self

    def __exit__(self, exc_type, exc, tb):
        for name, value in self.previous.items():
            if name in PARAMETER_TARGETS:
                set_parameter_value(name, value)
            else:
                globals()[name] = value
        refresh_runtime_paths()
        return False


def _sanitize_run_label(label):
    if label is None or str(label).strip() == "":
        return None
    text = str(label).strip()
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in text)[:120]


def refresh_runtime_paths():
    global FINAL_TREE_FILE, SURFACE_FILE, PRECOMPUTED_VTK, OUTPUT_DIR, TREE_FILE
    FINAL_TREE_FILE = MODEL_DIR / "final_tree.vtk"
    SURFACE_FILE = MODEL_DIR / "surface.nii.gz"
    PRECOMPUTED_VTK = MODEL_DIR / "data" / f"kirchhoff_p_in = {int(P_IN_MMHG)}.vtk"
    run_label = _sanitize_run_label(globals().get("RUN_LABEL", None))
    output_name = f"{int(P_IN_MMHG)}mmHg_{'myogenic' if ONLY_MYOGENIC else 'full'}"
    if run_label:
        output_name = f"{output_name}_{run_label}"
    OUTPUT_DIR = MODEL_OUTPUT_ROOT / output_name
    TREE_FILE = PRECOMPUTED_VTK if PRECOMPUTED_VTK.exists() else FINAL_TREE_FILE
    return {
        "final_tree_file": str(FINAL_TREE_FILE),
        "surface_file": str(SURFACE_FILE),
        "tree_file": str(TREE_FILE),
        "output_dir": str(OUTPUT_DIR),
    }


def build_model_from_parameter_values(parameter_values=None, parameter_table=None):
    # Xu stores parameters in module-level globals/classes rather than a nested
    # model dictionary. Return a normalized parameter dictionary for logging.
    if parameter_table is None:
        parameter_table = load_parameter_table()
    values = dict(BASELINE_PARAMETER_VALUES)
    if parameter_values:
        for key, value in parameter_values.items():
            if isinstance(key, int):
                matches = parameter_table.loc[parameter_table["input_index"] == key, "parameter_name"]
                if matches.empty:
                    raise KeyError(f"Unknown Xu 44 parameter index: {key}")
                key = matches.iloc[0]
            if key not in PARAMETER_TARGETS:
                raise KeyError(f"Unknown Xu 44 parameter: {key}")
            values[key] = float(value)
    return values


def terminal_results(auto):
    rows = []
    for terminal, (parent, child) in zip(auto.all_terminals, auto.all_terminal_edges):
        edge = auto.vt.tree[parent][child]
        rows.append({
            "terminal_node_id": terminal,
            "afferent_radius_um": edge.get("radius", np.nan),
            "afferent_flow_nl_s": edge.get("flow_from_Kirchhoff_nl/s", np.nan),
            "P_AA_in_mmHg": auto.vt.tree.nodes[parent].get("pressure_from_Kirchhoff_mmHg", np.nan),
            "P_GC_mmHg": auto.vt.tree.nodes[child].get("pressure_from_Kirchhoff_mmHg", np.nan),
            "SNGFR_nl_min": edge.get("filtration_rate", np.nan),
            "Cs_md_mM": edge.get("Cs_md", np.nan),
            "x_myo": edge.get("x_myo", np.nan),
            "x_tgf": edge.get("x_tgf", np.nan),
        })
    return pd.DataFrame(rows)


def run_model_instance(parameter_values=None, run_full_model=DEFAULT_RUN_FULL_MODEL, return_details=False, run_label=None, max_workers=None):
    values = build_model_from_parameter_values(parameter_values)
    previous_run_label = globals().get("RUN_LABEL", None)
    previous_max_workers = globals().get("MAX_WORKERS", None)
    if run_label is not None:
        globals()["RUN_LABEL"] = run_label
    if max_workers is not None:
        globals()["MAX_WORKERS"] = int(max_workers)
    try:
        context = XuParameterContext(values)
        context.__enter__()
        started = time.perf_counter()
        if run_full_model:
            auto, summary = run_xu_model_gsa()
            terminals = terminal_results(auto)
            total_gfr = float(terminals["SNGFR_nl_min"].sum())
            output = {
                **summary,
                "total_GFR_nl_min": total_gfr,
                "mean_SNGFR_nl_min": float(terminals["SNGFR_nl_min"].mean()),
                "median_SNGFR_nl_min": float(terminals["SNGFR_nl_min"].median()),
                "runtime_seconds": time.perf_counter() - started,
                "status": "success",
            }
            if return_details:
                return output, {"auto": auto, "terminals": terminals, "parameter_values": values}
            return output

        quick = simu(Q=4.0e6, r_v=10.0, P_t_in=65.0, P_GC=58.0, type=0, only_myo=False)
        output = {
            "status": "smoke_success",
            "run_full_model": False,
            "single_nephron_radius_um": float(quick[0]),
            "single_nephron_SNGFR_nl_min": float(quick[1]),
            "single_nephron_Cs_md_mM": float(quick[4]),
            "total_GFR_nl_min": np.nan,
            "runtime_seconds": time.perf_counter() - started,
        }
        if return_details:
            return output, {"single_nephron_result": quick, "parameter_values": values}
        return output
    finally:
        try:
            context.__exit__(None, None, None)
        finally:
            globals()["RUN_LABEL"] = previous_run_label
            if previous_max_workers is not None:
                globals()["MAX_WORKERS"] = previous_max_workers
            refresh_runtime_paths()


def run_xu_model_gsa():
    """Run one complete Xu pressure case and return solver plus GSA summary."""
    if not USE_AUTOREGULATION:
        raise NotImplementedError("No-autoregulation mode is a structural comparator and is not implemented in the 44-D forward interface.")
    if not TREE_FILE.exists():
        raise FileNotFoundError(TREE_FILE)
    if USE_NEPHRON_POPULATIONS and not SURFACE_FILE.exists():
        raise FileNotFoundError(SURFACE_FILE)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    auto = AutoRegulation(
        P_in=P_IN_MMHG,
        lr=LEARNING_RATE,
        only_myo=ONLY_MYOGENIC,
        num_iter=NUM_ITER,
        path=str(OUTPUT_DIR),
        pop=USE_NEPHRON_POPULATIONS,
        relTol=RELATIVE_FLOW_TOLERANCE,
    )
    auto.set_up_tree(
        root_loc=ROOT_LOCATION,
        mu=BLOOD_VISCOSITY,
        vspace=VOXEL_SIZE_UM,
        pt_file=str(TREE_FILE),
        surface_name=str(SURFACE_FILE),
    )
    started = time.perf_counter()
    before, after = auto.auto_reg()
    elapsed = time.perf_counter() - started
    summary = {
        "pressure_mmHg": float(P_IN_MMHG),
        "mode": "myogenic" if ONLY_MYOGENIC else "full",
        "use_nephron_populations": bool(USE_NEPHRON_POPULATIONS),
        "RBF_before_ml_min": float(before),
        "RBF_after_ml_min": float(after),
        "relative_RBF": float(after / before),
        "elapsed_seconds": float(elapsed),
        "terminal_count": int(len(auto.all_terminals)),
    }
    (OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return auto, summary


def run_forward_model(parameter_values=None, parameter_table=None, run_full_model=DEFAULT_RUN_FULL_MODEL, return_details=False, run_label=None, max_workers=None):
    values = build_model_from_parameter_values(parameter_values, parameter_table)
    return run_model_instance(
        values,
        run_full_model=run_full_model,
        return_details=return_details,
        run_label=run_label,
        max_workers=max_workers,
    )


GSA_PARAMETER_TABLE = load_parameter_table()
PARAMETER_VALIDATION = validate_parameter_table_against_model(GSA_PARAMETER_TABLE)
print("Grouped parameters loaded:", len(GSA_PARAMETER_TABLE))
print("All 44 recognized and baseline-matched:", bool(PARAMETER_VALIDATION["baseline_matches_model"].all()))

In [ ]:
baseline_smoke_check = run_forward_model(return_details=False, run_full_model=RUN_BASELINE_FULL_MODEL)
baseline_smoke_check

In [ ]:
def make_oat_grid(parameter_table=None, points_per_parameter=5, use_final_bounds=True):
    if parameter_table is None:
        parameter_table = load_parameter_table()
    lo_col = "final_lower" if use_final_bounds else "initial_lower"
    hi_col = "final_upper" if use_final_bounds else "initial_upper"
    rows = []
    for _, row in parameter_table.iterrows():
        values = np.linspace(float(row[lo_col]), float(row[hi_col]), points_per_parameter)
        for value in values:
            rows.append({
                "input_index": int(row["input_index"]),
                "parameter_name": row["parameter_name"],
                "value": float(value),
                "baseline": float(row["baseline"]),
                "lower": float(row[lo_col]),
                "upper": float(row[hi_col]),
            })
    return pd.DataFrame(rows)


def run_oat_smoke(points_per_parameter=3, limit=None):
    grid = make_oat_grid(points_per_parameter=points_per_parameter)
    if limit is not None:
        grid = grid.head(limit)
    rows = []
    for _, item in grid.iterrows():
        values = {item["parameter_name"]: item["value"]}
        started = time.perf_counter()
        try:
            out = run_forward_model(values, run_full_model=False)
            status = out.get("status", "success")
            error = ""
        except Exception as exc:
            out = {}
            status = "error"
            error = repr(exc)
        rows.append({
            **item.to_dict(),
            "status": status,
            "error": error,
            "total_GFR_nl_min": out.get("total_GFR_nl_min", np.nan),
            "single_nephron_SNGFR_nl_min": out.get("single_nephron_SNGFR_nl_min", np.nan),
            "single_nephron_Cs_md_mM": out.get("single_nephron_Cs_md_mM", np.nan),
            "runtime_seconds": time.perf_counter() - started,
        })
    return pd.DataFrame(rows)


print("OAT grid rows at 3 points/parameter:", len(make_oat_grid(points_per_parameter=3)))